In [ ]:
# V5 migration safety stop; see V5/README.md
raise RuntimeError('V5 MIGRATION INCOMPLETE: legacy V3.1 frozen satellite-reference and V4 intermediate data still required. Do not execute before dependency audit and V5-native inputs pass.')


# Musanga NB08 v4_005 — spectral detectability beyond forest-condition proxies

Retains all previous V3.1 frozen-reference, multiscale, NICFI, structure and DSM analyses (Sections 00–13). New **Sections 14–16** carry out a pre-specified, forest-stratified matched spectral contrast on exactly the existing 07A forest-condition sample. The primary hypothesis is higher **S2 B8 NIR reflectance** in Musanga-rich (≥40%) versus Musanga-poor (≤5%) 50 m cells **after matching observed forest conditions**. Other physical bands and NDVI are secondary and explicitly exploratory. Matching uses site, modelled canopy height, forest history, and tree cover only; satellite reflectance and Musanga fractions are NEVER used to compute match distances (labels only define the two groups). One-to-one low-cover controls, calipers, attrition and independent spatial uncertainty reporting. Yangambi may be non-estimable due to lack of high-cover cells.

These observations come from observational imagery; conditional spectral association is *not* species uniqueness or proof of physiological causation. Without reference labels for alternative secondary-tree species, unique Musanga identification remains untested. Height is an optical satellite model (2020); disturbance is JRC status 2020; Copernicus tree cover is 2019 at 100 m. All temporal/spatial mismatches apply.

**Run:** Section 00 and **Sections 14–16** (new, standalone after 00). Section 12 can be independently run but is not required. No Earth Engine, GPU or new downloads.


**V4 recovery:** run Section 00V4 immediately after Section 00 to replace archived UAV labels with the completed NB07 V4 full-ROI raster values. This preserves every historical scientific section; interpretation of native multiscale Sections 07–10 and 19–20 requires a separate V4 10m teacher regeneration.


## 00 — Configuration and exact cached reference audit

In [ ]:
%pip -q install pyarrow scikit-learn scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
ROOTS=[
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
 Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
found=[p for p in ROOTS if (p/'frozen_50m_reference.parquet').is_file()]
assert len(found)==1,('Expected one existing frozen reference; no new downloads',found)
ROOT=found[0];OUT=ROOT/'17_NB08_V3_1_detectability'
OUT.mkdir(parents=True,exist_ok=True)
D=pd.read_parquet(ROOT/'frozen_50m_reference.parquet')
SITES=('Baego','MOS','Yangambi','Yoko')
KEYS=['site','mosaic','row','col']
assert len(D)==14454 and not D.duplicated(KEYS).any()
assert D.groupby('site').size().to_dict()=={'Baego':480,'MOS':11494,'Yangambi':1383,'Yoko':1097}
assert D.target_fraction.between(0,1).all()
assert {'x','y','target_fraction','site'}.issubset(D.columns)
ANNUAL=sorted(c for c in D if c.startswith('RAW_ANNUAL__'))
QUARTER=sorted(c for c in D if c.startswith('S2_QUARTERLY_3YR__'))
EMB=sorted(c for c in D if c.startswith('EMB_ANNUAL__'))
assert len(ANNUAL)>0 and len(QUARTER)==44 and len(EMB)==64,(len(ANNUAL),len(QUARTER),len(EMB))
# Quarterly clear-observation counts are not spectral reflectance.
Q_COUNTS=[f'S2_QUARTERLY_3YR__{b:02d}' for b in (11,22,33,44)]
assert set(Q_COUNTS).issubset(QUARTER)
S2=[*ANNUAL,*[c for c in QUARTER if c not in Q_COUNTS]]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':[ *S2,*EMB ]}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
assert np.isfinite(D[ALL_FEATURES+['target_fraction','x','y']].to_numpy('float32')).all()
assert (D[Q_COUNTS].to_numpy(float)>=1).all(), 'Frozen reference has quarterly no-observation cells; inspect source.'
SEED=20261120
CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
           min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
# Compute fingerprints once; do not overwrite unrelated historical outputs.
SOURCE_SHA=hashlib.sha256(pd.util.hash_pandas_object(
 D[KEYS+['x','y','target_fraction']+ALL_FEATURES],index=False).values.tobytes()).hexdigest()
CONFIG={'version':'NB08_V3_1_LOFO_detectability_v4_001','sites':SITES,
 'n':len(D),'features':{k:len(v) for k,v in FEATURE_SETS.items()},
 'seed':SEED,'cap':CAP_PER_TRAIN_FOREST,'model':MODEL,
 'thresholds':THRESHOLDS,'source_sha256':SOURCE_SHA}
SIGNATURE=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:14]
RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
config_path=RUN/'config.json'
if config_path.is_file():assert json.loads(config_path.read_text())==json.loads(json.dumps(CONFIG))
else:config_path.write_text(json.dumps(CONFIG,indent=2))
print('Source:',ROOT/'frozen_50m_reference.parquet')
print('Saved outputs:',RUN,'| signature:',SIGNATURE)
display(D.groupby('site').target_fraction.agg(n='size',mean='mean',std='std',maximum='max').round(3))
print('Predictors: S2 annual',len(ANNUAL),'+ quarterly reflectance',len(QUARTER)-len(Q_COUNTS),'; embeddings',len(EMB))
print('No imagery downloads or raster reads are required.')


## 00V4 — V4 binary canopy fraction at historical 50 m reference locations

Run after Section 00. Band 2 is the binary fractional-cover source; band 1 mean probability is sensitivity only. The 2 m lattice aligns to exact 50 m cell boundaries, but native-area edge sensitivity remains to be checked. Six NB07 mosaic reconstruction reports must pass provenance checks. The V3.1 reference table supplies predictors and geographic keys, not V4 target values.


In [ ]:
# BLOCK 00V4 — FAST TILED V4 NB07 reference extraction, resumable by forest
# Uses the six existing NB07 rasters; no GPU, inference, downloads or random sampling.
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from contextlib import ExitStack

V4FINAL=ROOT.parents[3] if ROOT.parents[3].name=='Final' else next(
 (p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert V4FINAL is not None and V4FINAL.is_dir()
V4_MOSAICS=V4FINAL/'Outputs/v4/07_RGB_MS_100m_ROI_complete_sensors_v4/reconstructed_mosaics'
V4_GROUPS={'Baego':['Baego'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef',
        'Mos_3_4_transparent_mosaic_group1_georef',
        'Mos_5_6_transparent_mosaic_group1_georef'],
 'Yangambi':['Yangambi_georef'],'Yoko':['Yoko_kvo_georef']}
V4_PATHS={s:[V4_MOSAICS/f'{g}_Musanga_probability_binary_v4.tif' for g in gs]
          for s,gs in V4_GROUPS.items()}
assert all(p.is_file() for ps in V4_PATHS.values() for p in ps),V4_PATHS
V4_OUT=V4FINAL/'Outputs/v4/08_satellite_science_restored'
V4_OUT.mkdir(parents=True,exist_ok=True)
V4_SUPPORT=.90
V4_SIZE_M=50.
# A 2 m aggregation grid gives 25 x 25 support samples per 50 m cell.
# Edge support is approximate; native-area edge validation is needed for publication.
V4_SAMPLE_M=2.0
V4_TILE_M=500.0
V4_SOURCE_META={}
for paths in V4_PATHS.values():
    for p in paths:
        report=p.with_name(p.name.replace('_Musanga_probability_binary_v4.tif','_reconstruction.json'))
        assert report.is_file(),('NB07 reconstruction report missing',report)
        audit=json.loads(report.read_text())
        assert audit.get('status')=='complete' and audit.get('model')=='836ab7d77c1c' and audit.get('policy')=='47890f94c6cf',(p,audit)
        V4_SOURCE_META[str(p)]={'bytes':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns,'digest':audit.get('digest')}
V4_ID=hashlib.sha256(json.dumps({'inputs':V4_SOURCE_META,'size':V4_SIZE_M,
 'sample_m':V4_SAMPLE_M,'tile_m':V4_TILE_M,'method':'tiled_warp_v3_exact50m_footprint',
 'support':V4_SUPPORT,'target_band':2,'reference_key':SOURCE_SHA},
 sort_keys=True).encode()).hexdigest()[:16]
V4_CACHE=V4_OUT/f'00V4_retargeted_reference_{V4_ID}.parquet'

def _extract_forest_fast(forest, ref):
    year={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}[forest]
    annual=ROOT.parent.parent/'04_satellite_predictors/input_layers/RAW_ANNUAL'/(
        f'{forest}_RAW_ANNUAL_{year}_10m.tif')
    assert annual.is_file(),('Missing CRS reference',annual)
    xx=ref.x.to_numpy(float); yy=ref.y.to_numpy(float)
    # Tile all coordinates so each VRT window is read only once per raster/tile.
    tile_x=np.floor(xx/V4_TILE_M).astype('int64')
    tile_y=np.floor(yy/V4_TILE_M).astype('int64')
    tile_ids=pd.DataFrame({'tx':tile_x,'ty':tile_y},index=np.arange(len(ref)))
    tile_groups=list(tile_ids.groupby(['tx','ty'],sort=True).indices.values())
    best_valid=np.zeros(len(ref),dtype=np.float32)
    best_prob=np.full(len(ref),np.nan,dtype=np.float32)
    best_cover=np.full(len(ref),np.nan,dtype=np.float32)
    best_idx=np.full(len(ref),-1,dtype=np.int16)
    # Build a light 2 m virtual grid across only the reference locations.
    x0=float(xx.min()-25); top=float(yy.max()+25)
    w=int(np.ceil((float(xx.max())+25-x0)/V4_SAMPLE_M))
    h=int(np.ceil((top-(float(yy.min())-25))/V4_SAMPLE_M))
    tf=from_origin(x0,top,V4_SAMPLE_M,V4_SAMPLE_M)
    # Exact cell footprint mapped to destination pixel intervals.
    c0=np.maximum(0,np.floor((xx-25-x0)/V4_SAMPLE_M).astype(int))
    c1=np.minimum(w,np.ceil((xx+25-x0)/V4_SAMPLE_M).astype(int))
    r0=np.maximum(0,np.floor((top-(yy+25))/V4_SAMPLE_M).astype(int))
    r1=np.minimum(h,np.ceil((top-(yy-25))/V4_SAMPLE_M).astype(int))
    with ExitStack() as stack:
        anchor=stack.enter_context(rasterio.open(annual))
        assert anchor.crs is not None and anchor.crs.is_projected,forest
        layers=[]
        for p in V4_PATHS[forest]:
            ds=stack.enter_context(rasterio.open(p))
            assert ds.count==2 and ds.nodata==-9999 and ds.crs is not None,p
            layers.append(stack.enter_context(WarpedVRT(
                ds,crs=anchor.crs,transform=tf,width=w,height=h,
                resampling=Resampling.average,nodata=-9999)))
        print(f'  {forest}: {len(ref):,} cells, {len(tile_groups)} tiles, {len(layers)} mosaic(s)',flush=True)
        tstart=time.monotonic()
        for tnum,ids in enumerate(tile_groups,1):
            ids=np.asarray(ids,dtype=int)
            left=int(c0[ids].min());right=int(c1[ids].max())
            upper=int(r0[ids].min());lower=int(r1[ids].max())
            if right<=left or lower<=upper:continue
            win=Window(left,upper,right-left,lower-upper)
            for k,layer in enumerate(layers):
                a=layer.read([1,2],window=win,masked=True)
                v=np.logical_and(~np.ma.getmaskarray(a[0]),~np.ma.getmaskarray(a[1]))
                if not np.any(v):continue
                p=np.asarray(a[0].filled(np.nan),dtype=np.float32)
                f=np.asarray(a[1].filled(np.nan),dtype=np.float32)
                good=v & np.isfinite(p)&np.isfinite(f)&(p>=0)&(p<=1)&(f>=0)&(f<=1)
                for j in ids:
                    sl=(slice(r0[j]-upper,r1[j]-upper),slice(c0[j]-left,c1[j]-left))
                    g=good[sl]
                    frac=float(np.mean(g)) if g.size else 0.
                    if frac<V4_SUPPORT or frac<=best_valid[j]:continue
                    best_valid[j]=frac
                    best_prob[j]=float(np.mean(p[sl][g]))
                    best_cover[j]=float(np.mean(f[sl][g]))
                    best_idx[j]=k
            if tnum%20==0 or tnum==len(tile_groups):
                print(f'    {forest}: {tnum}/{len(tile_groups)} tiles; '
                      f'{int((best_valid>=V4_SUPPORT).sum()):,} valid; '
                      f'{(time.monotonic()-tstart)/60:.1f} min',flush=True)
    out=ref.copy()
    out['v4_mean_probability']=best_prob
    out['v4_valid_fraction']=best_valid
    out['v4_mosaic_index']=best_idx
    out['v4_target_definition']='NB07_binary_fraction_2m'
    out['target_fraction']=best_cover
    return out

if V4_CACHE.is_file():
    D4=pd.read_parquet(V4_CACHE)
    print('Reusing complete V4 reference cache:',V4_CACHE)
else:
    parts=[]
    for forest in SITES:
        part_file=V4_OUT/f'00V4_{V4_ID}_{forest}_checkpoint.parquet'
        if part_file.is_file():
            part=pd.read_parquet(part_file)
            print('Resuming checkpoint',forest,len(part),flush=True)
        else:
            ref=D[D.site.eq(forest)].copy().reset_index(drop=True)
            part=_extract_forest_fast(forest,ref)
            temp=part_file.with_suffix('.partial')
            part.to_parquet(temp,index=False)
            os.replace(temp,part_file)
            print('Checkpoint saved:',part_file,flush=True)
        parts.append(part)
    D4=pd.concat(parts,ignore_index=True)
    D4=D4[np.isfinite(D4.target_fraction)&D4.v4_valid_fraction.ge(V4_SUPPORT)].reset_index(drop=True)
    assert D4.site.nunique()==4,('V4 support missing in one forest',D4.site.value_counts().to_dict())
    assert not D4.duplicated(KEYS).any()
    temp=V4_CACHE.with_suffix('.partial')
    D4.to_parquet(temp,index=False);os.replace(temp,V4_CACHE)
print('V4 source-derived retained 50 m reference cells (historical locations):')
display(D4.groupby('site').agg(n=('target_fraction','size'),
 cover_pct=('target_fraction',lambda a:100*a.mean()),
 valid_pct=('v4_valid_fraction',lambda a:100*a.mean())).round(2))
D=D4
OUT=V4_OUT
SOURCE_SHA=hashlib.sha256(pd.util.hash_pandas_object(
 D[KEYS+['x','y','target_fraction','v4_mean_probability']+ALL_FEATURES],index=False).values.tobytes()).hexdigest()
CONFIG={**CONFIG,'version':'NB08_V4_NB07_label_retarget_tiled','n':len(D),
 'target':'NB07_V4_binary_band2_fraction','v4_id':V4_ID,'source_sha256':SOURCE_SHA}
SIGNATURE=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:14]
RUN=OUT/SIGNATURE;RUN.mkdir(parents=True,exist_ok=True)
(RUN/'config.json').write_text(json.dumps(CONFIG,indent=2))
y=D.target_fraction.to_numpy(np.float32);site=D.site.to_numpy()
print('V4 output:',RUN)
print('CAUTION: 2 m sampling approximates native-area edge coverage; review edge cells.')
print('Historical V3_1 target values excluded; no new V4 ROI cells added.')

assert D.target_fraction.between(0,1).all() and D.v4_valid_fraction.ge(V4_SUPPORT).all()
print('V4 binary-cover reference active; legacy V3.1 targets excluded.')


## 01 — A strict, small, cached leave-one-forest-out experiment

Use the original V3.1 source's **balanced-by-forest and cover-stratum** training principle; training selections depend only on training-forest targets. Each held-out forest is untouched until scoring. Single-model fits are intended as a first detectability test rather than hyperparameter optimization.

In [ ]:
y=D.target_fraction.to_numpy(np.float32)
site=D.site.to_numpy()
def balanced_draw(pool,heldout):
 rng=np.random.default_rng(SEED+sum(map(ord,heldout)))
 chosen=[]
 for forest in SITES:
  ix=pool[site[pool]==forest]
  if not len(ix):continue
  cap=min(len(ix),CAP_PER_TRAIN_FOREST)
  if len(ix)<=cap:
   chosen.extend(ix.tolist());continue
  strata=np.digitize(y[ix],[.10,.20,.40])
  bins={j:ix[strata==j] for j in range(4)}
  pick=[]
  # Keep up to a quarter of training sample above 40% where available.
  take=min(len(bins[3]),int(np.ceil(.25*cap)))
  if take:
   ids=rng.choice(bins[3],take,replace=False)
   pick.extend(ids.tolist());bins[3]=np.setdiff1d(bins[3],ids)
  while len(pick)<cap:
   active=[k for k in bins if len(bins[k])]
   assert active
   for k in active:
    n=min(len(bins[k]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
    if n>0:
     ids=rng.choice(bins[k],n,replace=False)
     pick.extend(ids.tolist());bins[k]=np.setdiff1d(bins[k],ids)
    if len(pick)>=cap:break
  chosen.extend(pick)
 return np.asarray(sorted(chosen),dtype=int)
predictions=[]
audit=[]
for held in SITES:
 test=np.flatnonzero(site==held)
 train_pool=np.flatnonzero(site!=held)
 train=balanced_draw(train_pool,held)
 assert not np.intersect1d(train,test).size
 audit.append(dict(heldout=held,train_cells=len(train),test_cells=len(test),
                   high_train=int((y[train]>=.40).sum())))
 # The constant null is learned from the **same** training draw.
 null_value=float(np.mean(y[train]))
 frame=D.iloc[test][KEYS+['x','y','target_fraction']].copy().reset_index(drop=True)
 frame['heldout']=held
 frame['M0_mean']=null_value
 for model_name,cols in FEATURE_SETS.items():
  cache=RUN/f'{held}_{model_name}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test_index'],test)
    assert z['source_sha256'].item()==SOURCE_SHA
    pred=z['pred'].astype(np.float32)
   print('CACHE',held,model_name,flush=True)
  else:
   t=time.monotonic()
   model=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   model.fit(D.iloc[train][cols].to_numpy('float32'),y[train])
   pred=np.clip(model.predict(D.iloc[test][cols].to_numpy('float32')),0,1).astype(np.float32)
   tmp=cache.with_suffix('.partial')
   with open(tmp,'wb') as f:
    np.savez_compressed(f,test_index=test,pred=pred,source_sha256=SOURCE_SHA)
   os.replace(tmp,cache)
   print('FIT',held,model_name,'seconds',round(time.monotonic()-t,1),flush=True)
  frame[model_name]=pred
 predictions.append(frame)
P=pd.concat(predictions,ignore_index=True)
assert len(P)==len(D) and not P.duplicated(KEYS).any()
assert P[['M0_mean',*FEATURE_SETS]].notna().all().all()
P.to_parquet(RUN/'01_paired_LOFO_predictions.parquet',index=False)
pd.DataFrame(audit).to_csv(RUN/'01_training_fold_sizes.csv',index=False)
display(pd.DataFrame(audit))
print('All held-out predictions cached on the same 14,454 frozen reference cells.')


## 02 — Forest-equal continuous-cover prediction and abundance detection

Use per-forest equal-weight metrics, never global pixel weighting that allows MOS to dominate. Continuous regression: R², MAE, RMSE, bias and Spearman. For thresholds 5%, 20%, 40%, compute ROC-AUC and average precision from **continuous prediction scores**, plus prevalence and a training-derived constant-score baseline. Undefined AUC for missing positive or negative classes is reported as NaN (not zero).

In [ ]:
MODELS=['M0_mean',*FEATURE_SETS]
def metrics(g,col):
 obs=g.target_fraction.to_numpy(float)
 pred=g[col].to_numpy(float)
 err=pred-obs
 var=float(np.mean((obs-np.mean(obs))**2))
 rho=spearmanr(obs,pred).statistic if np.std(pred)>1e-12 and np.std(obs)>1e-12 else np.nan
 return dict(n=len(g),R2=float(1-np.mean(err**2)/var) if var>0 else np.nan,
             MAE_pp=float(100*np.mean(np.abs(err))),
             RMSE_pp=float(100*np.sqrt(np.mean(err**2))),
             bias_pp=float(100*np.mean(err)),
             spearman_rho=float(rho) if np.isfinite(rho) else np.nan)
PER=pd.DataFrame([dict(site=forest,model=model,**metrics(g,model))
 for forest,g in P.groupby('site') for model in MODELS])
PER.to_csv(RUN/'02_continuous_by_forest.csv',index=False)
SUMMARY=[]
for model in MODELS:
 sub=PER[PER.model.eq(model)]
 # Global equal-forest R² with pooled equal-forest target variance.
 means=P.groupby('site').target_fraction.mean()
 v=np.mean([np.mean((g.target_fraction-means[s])**2)+(means[s]-means.mean())**2
      for s,g in P.groupby('site')])
 mse=np.mean([np.mean((g[model]-g.target_fraction)**2) for s,g in P.groupby('site')])
 SUMMARY.append(dict(model=model,n_cells=len(P),n_forests=len(SITES),
  equal_forest_R2=float(1-mse/v) if v>0 else np.nan,
  equal_forest_MAE_pp=float(sub.MAE_pp.mean()),
  equal_forest_RMSE_pp=float(np.sqrt(np.mean(sub.RMSE_pp**2))),
  equal_forest_bias_pp=float(sub.bias_pp.mean()),
  mean_forest_rho=float(sub.spearman_rho.mean())))
SUMMARY=pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN/'02_equal_forest_summary.csv',index=False)
DISC=[]
for (forest,model),g in (( (f,m),q) for f,q in P.groupby('site') for m in MODELS):
 obs=g.target_fraction.to_numpy(float)
 pred=g[model].to_numpy(float)
 for t in THRESHOLDS:
  true=(obs>=t).astype(int)
  npos=int(true.sum());nneg=len(true)-npos
  auc=roc_auc_score(true,pred) if npos and nneg and np.unique(pred).size>1 else np.nan
  ap=average_precision_score(true,pred) if npos and nneg else np.nan
  DISC.append(dict(site=forest,model=model,threshold=t,
       n=len(g),positive_n=npos,negative_n=nneg,
       prevalence=npos/len(g),ROC_AUC=auc,average_precision=ap))
DISC=pd.DataFrame(DISC)
DISC.to_csv(RUN/'02_abundance_detection_by_forest.csv',index=False)
DISPLAY_DISC=(DISC.groupby(['model','threshold']).agg(
 n_forests_evaluable=('ROC_AUC','count'),
 mean_AUC=('ROC_AUC','mean'),
 mean_AP=('average_precision','mean'),
 mean_prevalence=('prevalence','mean')).reset_index())
DISPLAY_DISC.to_csv(RUN/'02_equal_forest_detection_summary.csv',index=False)
print('Equal-forest continuous-cover scores:')
display(SUMMARY.round(3))
print('Abundance discrimination (undefined sites omitted, counts explicit):')
display(DISPLAY_DISC.round(3))
print('Important: high-cover detection is undefined for forests without high-cover examples.')


## 03 — Visual evidence: spectral contrasts, held-out predictions and detectability

**Figure A:** within-forest low vs high cover spectral contrast on the frozen 50 m cells; avoids visually conflating broad differences among forests. Spectral columns are raw annual source bands (band numbers until independently verified); do not claim wavelengths without an audited metadata lookup.

**Figure B:** held-out continuous fractional cover by forest, preserving the four-fold transfer test.

**Figure C:** abundance discrimination curves at 5, 20 and 40% (mean of evaluable forests, number of evaluable forests displayed).

In [ ]:
import matplotlib
from matplotlib.lines import Line2D
# A: Within-forest annual S2 contrast. Use percentiles for robustness.
low=.05;high=.40
rows=[]
for forest,g in D.groupby('site'):
 lo=g[g.target_fraction<=low]
 hi=g[g.target_fraction>=high]
 if len(lo)<10 or len(hi)<10:
  print('Spectral contrast not estimable',forest,'low',len(lo),'high',len(hi));continue
 for i,c in enumerate(ANNUAL,1):
  lv=float(lo[c].median());hv=float(hi[c].median())
  rows.append(dict(site=forest,band=c,band_order=i,n_low=len(lo),
                   n_high=len(hi),median_low=lv,median_high=hv,
                   difference=hv-lv))
SPECTRA=pd.DataFrame(rows)
SPECTRA.to_csv(RUN/'03_within_forest_annual_S2_spectra.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,8),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=SPECTRA[SPECTRA.site.eq(forest)]
 if len(g):
  ax.plot(g.band_order,g.median_low,'o-',label='≤5% UAV Musanga')
  ax.plot(g.band_order,g.median_high,'o-',label='≥40% UAV Musanga')
  ax.set_title(f'{forest} | low n={g.n_low.iloc[0]}, high n={g.n_high.iloc[0]}')
 else:
  ax.set_title(f'{forest} | too few high/low cells')
 ax.set(xlabel='Annual S2 source band number',ylabel='Median archived band value')
 ax.grid(alpha=.15)
axs.flat[0].legend(fontsize=8)
fig.suptitle('A · Within-forest spectral associations (not species-specific identification)')
fig.savefig(RUN/'03A_S2_within_forest_low_high_spectra.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)
# B: For each forest show S2 vs embeddings+S2; cap plots only, not metrics.
fig,axs=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 g=P[P.site.eq(forest)].copy()
 if len(g)>1300:g=g.sample(n=1300,random_state=SEED)
 ax.scatter(g.target_fraction*100,g.M1_S2*100,s=9,alpha=.3,label='S2')
 ax.scatter(g.target_fraction*100,g.M3_S2_EMB*100,s=9,alpha=.3,label='S2 + embeddings')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlabel='UAV Musanga cover (%)',
        ylabel='Held-out satellite prediction (%)',
        xlim=(-2,102),ylim=(-2,102))
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('B · Strict leave-one-forest-out prediction (matched reference cells)')
fig.savefig(RUN/'03B_heldout_scatter_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# C: Mean AUC by cover threshold and number of forests actually evaluable.
fig,ax=plt.subplots(figsize=(8,5))
for model in MODELS[1:]:
 q=DISPLAY_DISC[DISPLAY_DISC.model.eq(model)].sort_values('threshold')
 ax.plot(q.threshold*100,q.mean_AUC,marker='o',label=model)
 for v in q.itertuples():
  if np.isfinite(v.mean_AUC):
   ax.annotate(f'n={v.n_forests_evaluable}',(v.threshold*100,v.mean_AUC),
               xytext=(0,8),textcoords='offset points',ha='center',fontsize=8)
ax.axhline(.5,ls='--',color='gray',lw=1,label='Chance AUC')
ax.set(xlabel='UAV Musanga cover threshold (%)',ylabel='Forest-mean held-out ROC-AUC',
       ylim=(0,1.05),xticks=[5,20,40],
       title='C · Ability to distinguish Musanga-abundant patches')
ax.legend(fontsize=9);ax.grid(alpha=.15)
fig.savefig(RUN/'03C_abundance_discrimination.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Three figures saved:',RUN)


## 04 — Compact decision and caveats

Interpret performance for **all four forests**, not just the pooled R². Compare M1 vs M0 (satellite information); M2 vs M1 (embeddings versus raw S2); and M3 vs both. AUC > 0.5 on evaluable, held-out forests supports **association**, not Musanga-specific spectral identifiability. Inspect forest abundance prevalence: at >40% the Yangambi fold has no positives, so the ROC-AUC is not defined.

**Caveats:** frozen V3.1 labels derive from an earlier UAV model; pixel-neighbour dependence invalidates naive cell-wise confidence intervals. Sampling to balance training forests is fixed, not evaluated as an optimal algorithm. The threshold experiment uses continuous regression scores—not binary models—avoiding separate threshold-fitting. Do not interpret within-forest band profiles as independent species spectra; examine confounding by young forest and disturbance in a later experiment.

**Next iteration:** once NB07 V4 RGB+MS inference is complete, reaggregate its valid-pixel predictions to the exact 50 m satellite cells, keep a matched validity mask, and repeat the same fixed experiment. Do not mix V3.1 and V4 targets in a single score.

In [ ]:
manifest={'experiment':'NB08_v4_001_V3_1','input':str(ROOT/'frozen_50m_reference.parquet'),
 'source_sha256':SOURCE_SHA,'signature':SIGNATURE,
 'n_reference_cells':len(D),'n_sites':len(SITES),
 'feature_families':{k:len(v) for k,v in FEATURE_SETS.items()},
 'spatial_validation':'leave_one_forest_out',
 'reference':'V3_1 UAV teacher-derived fractional Musanga cover',
 'warning':'Predictive association is not species-unique identification',
 'outputs':str(RUN)}
(RUN/'04_run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('NB08 V3.1 detection experiment completed:',RUN)
print('Primary output: 02_equal_forest_summary.csv')
print('Detection output: 02_equal_forest_detection_summary.csv')
print('Paired predictions: 01_paired_LOFO_predictions.parquet')


## 05 — Proper Sentinel-2 spectra and seasonal separability

Plot only reflectance bands at their physical wavelength locations, not NDVI/NDMI/counts. Within each forest, compare ≤5% versus ≥40% cover, showing IQR and effect sizes. Quarterly columns are arranged 10 reflectance channels + 1 count for each pooled 3-year quarter. These are not monthly observations.

In [ ]:
# 10 original annual reflectance bands (verified upstream Earth Engine selection order)
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
S2_WAVELENGTH_NM=np.array([492,560,665,833,704,740,783,865,1610,2190])
S2_ORDER=np.argsort(S2_WAVELENGTH_NM)
assert len(ANNUAL)==15 and len(QUARTER)==44
REFLECTANCE=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
INDICES=[f'RAW_ANNUAL__{i:02d}' for i in range(11,15)]
CLEAR_ANNUAL='RAW_ANNUAL__15'
Q_REFLECTANCE={q:[f'S2_QUARTERLY_3YR__{(q-1)*11+i:02d}' for i in range(1,11)] for q in range(1,5)}
assert set(REFLECTANCE+INDICES+[CLEAR_ANNUAL])==set(ANNUAL)
assert set(sum(Q_REFLECTANCE.values(),[])+Q_COUNTS)==set(QUARTER)
# Plot median and interquartile envelope of raw reflectance; no modelling in this block.
spectral_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(15,10),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast\\nlow={len(low)}, high={len(high)}',
          ha='center',va='center',transform=ax.transAxes)
  ax.set_title(f'{forest} — not estimable')
  continue
 for subset,label,color in [(low,'≤5% Musanga','steelblue'),(high,'≥40% Musanga','darkorange')]:
  arr=subset[REFLECTANCE].to_numpy(float)[:,S2_ORDER]
  wl=S2_WAVELENGTH_NM[S2_ORDER]
  med=np.median(arr,axis=0);q25=np.percentile(arr,25,axis=0);q75=np.percentile(arr,75,axis=0)
  ax.plot(wl,med,'o-',label=f'{label}, n={len(subset)}',color=color)
  ax.fill_between(wl,q25,q75,alpha=.16,color=color)
  for j,k in enumerate(S2_ORDER):
   spectral_rows.append(dict(site=forest,group=label,band=S2_BANDS[k],
     wavelength_nm=int(S2_WAVELENGTH_NM[k]),n=len(subset),
     median=float(med[j]),q25=float(q25[j]),q75=float(q75[j])))
 ax.set(xlabel='Approximate central wavelength (nm)',
        ylabel='S2 annual median surface reflectance',title=forest)
 ax.legend(fontsize=8);ax.grid(alpha=.12)
fig.suptitle('True annual S2 reflectance: low vs high 50 m Musanga cover (median ± IQR)')
fig.savefig(RUN/'05A_annual_S2_true_reflectance_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(spectral_rows).to_csv(RUN/'05A_annual_S2_true_reflectance_IQR.csv',index=False)
# Seasonal reflectance: 3-year pooled quarter medians, NOT single-year snapshots.
seasonal_rows=[]
fig,axarr=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for ax,forest in zip(axarr.flat,SITES):
 g=D[D.site.eq(forest)]
 low=g[g.target_fraction<=.05];high=g[g.target_fraction>=.40]
 if len(high)<10 or len(low)<10:
  ax.text(.5,.5,f'Insufficient contrast: high={len(high)}',ha='center',va='center',transform=ax.transAxes)
  ax.set_title(forest);continue
 for q in range(1,5):
  for subset,name in [(low,'low'),(high,'high')]:
   # NDVI is computed from quarter median reflectances (B8 = index 4; B4 = index 3).
   nir=subset[Q_REFLECTANCE[q][3]].to_numpy(float)
   red=subset[Q_REFLECTANCE[q][2]].to_numpy(float)
   ndvi=(nir-red)/np.maximum(nir+red,1e-8)
   med=float(np.median(ndvi))
   seasonal_rows.append(dict(site=forest,quarter=q,class_name=name,
                              n=len(subset),median_NDVI=med,
                              q25_NDVI=float(np.percentile(ndvi,25)),
                              q75_NDVI=float(np.percentile(ndvi,75))))
 for name,color in [('low','steelblue'),('high','darkorange')]:
  a=pd.DataFrame(seasonal_rows)
  a=a[(a.site==forest)&(a.class_name==name)].sort_values('quarter')
  ax.plot(a.quarter,a.median_NDVI,'o-',color=color,label=name)
  ax.fill_between(a.quarter.to_numpy(float),a.q25_NDVI.to_numpy(float),
                  a.q75_NDVI.to_numpy(float),alpha=.15,color=color)
 ax.set(title=forest,xlabel='Pooled 3-year calendar quarter',ylabel='Quarterly median-reflectance NDVI',
        xticks=[1,2,3,4])
 ax.grid(alpha=.15);ax.legend()
fig.suptitle('Seasonal spectral association: quarter medians pooled across three years')
fig.savefig(RUN/'05B_pooled_quarterly_NDVI_IQR.png',dpi=180)
plt.show();plt.close(fig)
pd.DataFrame(seasonal_rows).to_csv(RUN/'05B_pooled_quarterly_NDVI_IQR.csv',index=False)
print('Spectral plots complete. Overlap of distributions is expected in mixed 50 m canopies.')


## 06 — Why does Sentinel-2 predict so well? LOFO annual-versus-seasonal ablation

This isolates single-year reflectance, four annual indices, and 3-year pooled quarterly reflectance. Identical targets, same training draws (from Section 01), same holdouts and model hyperparameters. It does **not** infer phenological mechanisms from predictive performance alone. All metrics are computed with equal weight for each forest.

In [ ]:
ABLATION={
 'A_reflectance_only':REFLECTANCE,
 'B_annual_plus_indices':REFLECTANCE+INDICES,
 'C_pooled_quarters_only':sum(Q_REFLECTANCE.values(),[]),
 'D_annual_plus_quarters':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[]),
 'E_embeddings_only':EMB,
 'F_all_plus_embeddings':REFLECTANCE+INDICES+sum(Q_REFLECTANCE.values(),[])+EMB
}
ABL_CFG={'version':'NB08_S2_temporal_ablation_v1','seed':SEED,'model':MODEL,
        'train_cap':CAP_PER_TRAIN_FOREST,'features':ABLATION,
        'source':SOURCE_SHA}
ABL_ID=hashlib.sha256(json.dumps(ABL_CFG,sort_keys=True).encode()).hexdigest()[:12]
ABL_DIR=OUT/'18_S2_temporal_ablation'/ABL_ID
ABL_DIR.mkdir(parents=True,exist_ok=True)
(ABL_DIR/'config.json').write_text(json.dumps(ABL_CFG,indent=2))
rows=[];pred_frames=[]
for forest in SITES:
 test=np.flatnonzero(site==forest)
 train=balanced_draw(np.flatnonzero(site!=forest),forest)
 part=D.iloc[test][KEYS+['target_fraction']].copy().reset_index(drop=True)
 for variant,features in ABLATION.items():
  cache=ABL_DIR/f'{forest}_{variant}.npz'
  if cache.is_file():
   with np.load(cache,allow_pickle=False) as z:
    assert np.array_equal(z['test'],test)
    predict=z['pred'].astype(float)
   print('CACHE',forest,variant,flush=True)
  else:
   start=time.monotonic()
   fit=HistGradientBoostingRegressor(random_state=SEED,**MODEL)
   fit.fit(D.iloc[train][features].to_numpy(np.float32),y[train])
   predict=np.clip(fit.predict(D.iloc[test][features].to_numpy(np.float32)),0,1)
   with open(cache.with_suffix('.partial'),'wb') as f:
    np.savez_compressed(f,test=test,pred=predict.astype(np.float32))
   os.replace(cache.with_suffix('.partial'),cache)
   print('FIT',forest,variant,round(time.monotonic()-start,1),'seconds',flush=True)
  part[variant]=predict
  err=predict-y[test]
  rows.append(dict(site=forest,variant=variant,n=len(test),n_features=len(features),
     MAE_pp=float(100*np.mean(abs(err))),MSE=float(np.mean(err**2)),
     bias_pp=float(100*np.mean(err)),
     R2=float(1-np.mean(err**2)/np.var(y[test])) if np.var(y[test])>0 else np.nan))
 pred_frames.append(part)
ABL_P=pd.concat(pred_frames,ignore_index=True)
ABL_P.to_parquet(ABL_DIR/'06_all_paired_LOFO_predictions.parquet',index=False)
ABL_PER=pd.DataFrame(rows)
ABL_PER.to_csv(ABL_DIR/'06_per_forest.csv',index=False)
forest_mean=D.groupby('site').target_fraction.mean().mean()
target_variance=float(np.mean([
 np.mean((q.target_fraction.to_numpy(float)-forest_mean)**2)
 for _,q in D.groupby('site')]))
summary=ABL_PER.groupby('variant').agg(n_forests=('site','nunique'),
  mean_MAE_pp=('MAE_pp','mean'),mean_MSE=('MSE','mean'),
  mean_bias_pp=('bias_pp','mean'),mean_per_forest_R2=('R2','mean'))
summary['equal_forest_R2']=1-summary.mean_MSE/target_variance
summary['equal_forest_RMSE_pp']=100*np.sqrt(summary.mean_MSE)
summary=summary.reset_index()
summary.to_csv(ABL_DIR/'06_equal_forest_ablation_summary.csv',index=False)
display(summary[['variant','n_forests','mean_MAE_pp','equal_forest_R2',
                 'equal_forest_RMSE_pp','mean_bias_pp']].round(3))
display(ABL_PER.pivot(index='site',columns='variant',values='MAE_pp').round(2))
fig,ax=plt.subplots(figsize=(11,5),layout='constrained')
ax.bar(summary.variant,summary.equal_forest_R2)
ax.axhline(0,color='gray',lw=.8)
ax.tick_params(axis='x',rotation=28)
ax.set(ylabel='Equal-forest R²',title='Does seasonal S2 add transfer beyond annual reflectance and indices?')
fig.savefig(ABL_DIR/'06_S2_annual_vs_seasonal_LOFO.png',dpi=180)
plt.show();plt.close(fig)
print('Annual-versus-seasonal LOFO results:',ABL_DIR)


## 07 — Native V4 source audit and multiresolution readiness

The historical V3.1 10 m teacher is retained only for scientific provenance. V4 multiresolution targets must come independently from NB07 native binary mosaics. Do not publish previous V3.1 multiresolution outputs as V4 validation.


In [ ]:
assert 'V4_PATHS' in globals() and 'D4' in globals(), 'Run Section 00V4 first.'
# Read-only source discovery. Never silently infer multiscale targets from 50m cells.
from glob import glob
BASE=ROOT.parent
teacher_dir=ROOT/'11A_dense_UAV_teacher_10m'
predictor_dir=BASE.parent/'04_satellite_predictors'/'input_layers'
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
scale_rows=[]
for forest in SITES:
 teacher=teacher_dir/f'{forest}_Musanga_UAVteacher_mean_10m.tif'
 annual=predictor_dir/'RAW_ANNUAL'/f'{forest}_RAW_ANNUAL_{YEARS[forest]}_10m.tif'
 embeddings=predictor_dir/'EMB_ANNUAL'/f'{forest}_EMB_ANNUAL_{YEARS[forest]}_10m.tif'
 scale_rows.append(dict(site=forest,teacher_10m_present=teacher.is_file(),v4_mosaics_present=all(p.is_file() for p in V4_PATHS[forest]),
  annual_S2_10m_present=annual.is_file(),embedding_10m_present=embeddings.is_file(),
  teacher_file=str(teacher),S2_file=str(annual),embedding_file=str(embeddings)))
SCALE_INPUT_AUDIT=pd.DataFrame(scale_rows)
display(SCALE_INPUT_AUDIT[['site','teacher_10m_present','annual_S2_10m_present',
                            'embedding_10m_present']])
SCALE_INPUT_AUDIT.to_csv(RUN/'07_multiscale_source_audit.csv',index=False)
if not SCALE_INPUT_AUDIT[['teacher_10m_present','annual_S2_10m_present','embedding_10m_present']].all().all():
 print('Some sources are missing from expected paths. Locate/verify actual cache paths before multi-scale extraction.')
print('PROPOSED genuine analysis resolutions (m):', [10,20,25,30,50,75,100])
print('Scale benchmark NOT run. Need verified harmonized labels, support and identical spatial footprints.')

assert SCALE_INPUT_AUDIT.v4_mosaics_present.all(), 'Missing V4 source mosaics.'


## 08 — Independent V4 multiresolution source aggregation

Source is NB07 V4 high-resolution binary band 2, not the historical V3.1 teacher. Each grid is evaluated independently using an aligned 1 m support lattice (area approximation); 50 m agreement with Section 00V4 (2 m lattice) is audited. The scientific predictor and LOFO analyses downstream are retained. Code is unexecuted until Colab validation.


In [ ]:
assert 'V4_ID' in globals() and 'D4' in globals(), 'Run Section 00V4 first.'
assert D.equals(D4), 'The V4 label bridge is not active.'
import rasterio
from rasterio.warp import reproject
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.windows import Window
from contextlib import ExitStack
SCALES=[10,25,30,50,75,100] # Genuine target/predictor support, metres
MIN_TEACHER_COVERAGE=.90
MIN_SOURCE_S2_COVERAGE=.90
MAX_EXTRACT_GRID_PIXELS=9_000_000
SCALE_VERSION='NB08_V4_NB07_binary_multires_1m_sampling_v1'
SCALE_OUT=V4_OUT/'19_v4_native_scale_LOFO'
SCALE_OUT.mkdir(parents=True,exist_ok=True)
SCALE_PATHS={}
SCALE_YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
for forest in SITES:
 teacher_paths=V4_PATHS[forest]
 annual=ROOT.parent.parent/'04_satellite_predictors/input_layers/RAW_ANNUAL'/f'{forest}_RAW_ANNUAL_{SCALE_YEARS[forest]}_10m.tif'
 # The native S2 sources were already found in Section 07; explicit assertion prevents silent re-extraction.
 assert all(p.is_file() for p in teacher_paths),('Missing V4 NB07 mosaic',forest)
 assert annual.is_file(),('Missing annual S2',annual)
 SCALE_PATHS[forest]={'v4_mosaics':teacher_paths,'annual':annual}
manifest=dict(version=SCALE_VERSION,scales=SCALES,teacher_coverage=MIN_TEACHER_COVERAGE,
   s2_coverage=MIN_SOURCE_S2_COVERAGE,source_sha=SOURCE_SHA,
   v4_id=V4_ID,source_mosaics=V4_SOURCE_META,
   inputs={forest:{'annual':(str(paths['annual']),paths['annual'].stat().st_size,paths['annual'].stat().st_mtime_ns),
                   'v4_mosaics':[str(p) for p in paths['v4_mosaics']]}
           for forest,paths in SCALE_PATHS.items()})
SCALE_HASH=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:14]
SCALE_RUN=SCALE_OUT/SCALE_HASH
SCALE_RUN.mkdir(parents=True,exist_ok=True)
(SCALE_RUN/'08_source_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Native-scale output:',SCALE_RUN)

def mean_warp(src,dst_transform,h,w,index=1,source_array=None,source_nodata=None):
 target=np.full((h,w),np.nan,dtype=np.float32)
 if source_array is None:
  source=rasterio.band(src,index)
  source_transform=src.transform
  source_crs=src.crs
  if source_nodata is None:source_nodata=src.nodata
 else:
  source=source_array
  source_transform=src.transform
  source_crs=src.crs
 reproject(source,target,src_transform=source_transform,src_crs=source_crs,
   src_nodata=source_nodata,dst_transform=dst_transform,dst_crs=src.crs,
   dst_nodata=np.nan,resampling=Resampling.average,num_threads=2)
 return target

def valid_warp(src,dst_transform,h,w,index=1):
 # Average of binary valid-data indicators: fraction of real source support.
 data=src.read(index,masked=True)
 good=(~np.ma.getmaskarray(data)) & np.isfinite(np.asarray(data.filled(np.nan)))
 arr=good.astype('float32')
 out=np.zeros((h,w),np.float32)
 reproject(arr,out,src_transform=src.transform,src_crs=src.crs,
   src_nodata=None,dst_transform=dst_transform,dst_crs=src.crs,
   dst_nodata=0,resampling=Resampling.average,num_threads=2)
 return out

def build_scale_site(scale,forest):
 out=SCALE_RUN/f'{forest}_{scale}m.parquet'
 if out.is_file():
  cached=pd.read_parquet(out)
  assert cached.scale_m.eq(scale).all() and cached.site.eq(forest).all()
  assert (cached.v4_target_definition=='NB07_binary_fraction_native_1m_support').all()
  print('CACHE',forest,scale,'n',len(cached),flush=True)
  return cached
 ref=D[D.site.eq(forest)]
 rr=ref.row.to_numpy(int);cc=ref.col.to_numpy(int)
 with ExitStack() as stack:
  s2=stack.enter_context(rasterio.open(SCALE_PATHS[forest]['annual']))
  assert s2.crs is not None and s2.crs.is_projected,(forest,'Invalid S2 CRS')
  assert s2.count>=14,(forest,'annual S2 lacks reflectances/indices')
  assert abs(s2.res[0]-10)<.1 and abs(s2.res[1]-10)<.1,(forest,s2.res)

  # Each 50m frozen cell is addressed on the site's 10m Sentinel grid.
  factor=scale/10.
  r0=int(np.floor(rr.min()*50/scale));c0=int(np.floor(cc.min()*50/scale))
  r1=int(np.ceil((rr.max()+1)*50/scale));c1=int(np.ceil((cc.max()+1)*50/scale))
  h=r1-r0;w=c1-c0
  assert h*w<=MAX_EXTRACT_GRID_PIXELS,(forest,scale,'too large grid',h,w)
  dst_transform=s2.transform*Affine.translation(c0*factor,r0*factor)*Affine.scale(factor,factor)
  s2_support=valid_warp(s2,dst_transform,h,w)
  good=s2_support>=MIN_SOURCE_S2_COVERAGE
  # Use only the same approved 50m geographic cell footprints from the frozen reference.
  ref_width=int(cc.max()+1)
  keyset=np.unique(rr.astype(np.int64)*ref_width+cc)
  ygrid,xgrid=np.nonzero(good)
  base_row=np.floor((r0+ygrid+.5)*scale/50).astype(np.int64)
  base_col=np.floor((c0+xgrid+.5)*scale/50).astype(np.int64)
  eligible=np.isin(base_row*ref_width+base_col,keyset,assume_unique=False)
  ygrid=ygrid[eligible];xgrid=xgrid[eligible]
  base_row=base_row[eligible];base_col=base_col[eligible]
  assert len(ygrid)>0,(forest,scale,'No matched V4 locations')
  # V4 native mosaics are sampled independently for each resolution.
  # A 1m lattice is used to approximate fractional native area support,
  # not a resampled V3.1 teacher or pre-computed 50m reference.
  cover=np.full(len(ygrid),np.nan,np.float32)
  teach_support=np.zeros(len(ygrid),np.float32)
  source_index=np.full(len(ygrid),-1,np.int16)
  sample_factor=int(scale)
  fine_transform=dst_transform*Affine.scale(1/sample_factor,1/sample_factor)
  block=max(1,400//scale)
  tilegroups=pd.DataFrame({'ty':ygrid//block,'tx':xgrid//block}).groupby(['ty','tx'],sort=True).indices
  with ExitStack() as v4_stack:
   layers=[]
   for path in SCALE_PATHS[forest]['v4_mosaics']:
    src=v4_stack.enter_context(rasterio.open(path))
    assert src.count==2 and src.nodata==-9999 and src.crs is not None,path
    layers.append(v4_stack.enter_context(WarpedVRT(
      src,crs=s2.crs,transform=fine_transform,
      width=w*sample_factor,height=h*sample_factor,
      resampling=Resampling.average,nodata=-9999)))
   for chunk,ids in enumerate(tilegroups.values(),1):
    ids=np.asarray(ids,dtype=int)
    y0=int(ygrid[ids].min());y1=int(ygrid[ids].max()+1)
    x0=int(xgrid[ids].min());x1=int(xgrid[ids].max()+1)
    nr=y1-y0;nc=x1-x0
    window=Window(x0*sample_factor,y0*sample_factor,
                  nc*sample_factor,nr*sample_factor)
    ir=ygrid[ids]-y0;ic=xgrid[ids]-x0
    for k,layer in enumerate(layers):
     a=layer.read(2,window=window,masked=True)
     z=np.asarray(a.filled(np.nan),dtype=np.float32)
     valid=(~np.ma.getmaskarray(a))&np.isfinite(z)&(z>=0)&(z<=1)
     nvalid=valid.reshape(nr,sample_factor,nc,sample_factor).sum(axis=(1,3))
     totals=np.where(valid,z,0).reshape(nr,sample_factor,nc,sample_factor).sum(axis=(1,3))
     support=(nvalid[ir,ic]/float(sample_factor*sample_factor)).astype(np.float32)
     selected=(support>=MIN_TEACHER_COVERAGE)&(support>teach_support[ids])
     if np.any(selected):
      subset=ids[selected]
      teach_support[subset]=support[selected]
      cover[subset]=(totals[ir[selected],ic[selected]]/nvalid[ir[selected],ic[selected]]).astype(np.float32)
      source_index[subset]=k
    if chunk%50==0 or chunk==len(tilegroups):
     print(f'V4 {forest} {scale}m source tiles {chunk}/{len(tilegroups)}',flush=True)
  valid_target=np.isfinite(cover)&(teach_support>=MIN_TEACHER_COVERAGE)
  ygrid=ygrid[valid_target];xgrid=xgrid[valid_target]
  base_row=base_row[valid_target];base_col=base_col[valid_target]
  cover=cover[valid_target];teach_support=teach_support[valid_target]
  source_index=source_index[valid_target]
  assert len(cover)>0,(forest,scale,'No V4-supported cells')

  # 14 annual reflectance and vegetation indices, computed by true area aggregation.
  feature_data={}
  for j in range(1,15):
   image=mean_warp(s2,dst_transform,h,w,index=j)
   # Select only actual model cells; exclude nodata and outside-source areas.
   feature_data[f'S2_{j:02d}']=image[ygrid,xgrid].copy()
   del image
  df=pd.DataFrame(dict(site=forest,scale_m=scale,row=r0+ygrid,col=c0+xgrid,
     frozen_50m_row=base_row,frozen_50m_col=base_col,
     x=np.asarray((dst_transform*(xgrid+.5,ygrid+.5))[0],float),
     y=np.asarray((dst_transform*(xgrid+.5,ygrid+.5))[1],float),
     target_fraction=cover,
     teacher_coverage=teach_support,v4_mosaic_index=source_index,
     v4_target_definition='NB07_binary_fraction_native_1m_support',
     s2_coverage=s2_support[ygrid,xgrid],**feature_data))
  df=df[np.isfinite(df[[f'S2_{j:02d}' for j in range(1,15)]].to_numpy(float)).all(axis=1)].copy()
  # Physical reflectances must not be nodata or invalid (indices can be negative).
  df=df[(df[[f'S2_{j:02d}' for j in range(1,11)]]>=0).all(axis=1)].reset_index(drop=True)
  assert df.target_fraction.between(0,1).all() and not df.duplicated(['site','row','col']).any()
  df.to_parquet(out,index=False)
  print('SAVED',forest,scale,'n',len(df),'support>=' ,MIN_TEACHER_COVERAGE,flush=True)
  return df

# Always audit 50m reconstructed teacher BEFORE any other scale, using exact grid keys.
AUDIT50=[]
for forest in SITES:
 f=build_scale_site(50,forest)
 orig=D[D.site.eq(forest)][KEYS+['target_fraction']]
 joined=orig.merge(f,left_on=['site','row','col'],right_on=['site','row','col'],
                    how='inner',suffixes=('_frozen','_native'))
 assert len(joined)>=.90*len(orig),(forest,'V4 support overlap too low',len(joined),len(orig))
 diff=joined.target_fraction_native.to_numpy()-joined.target_fraction_frozen.to_numpy()
 mae=float(np.mean(abs(diff))*100)
 AUDIT50.append(dict(site=forest,n=len(joined),mae_pp=mae,
     bias_pp=float(np.mean(diff)*100),corr=float(np.corrcoef(
       joined.target_fraction_native,joined.target_fraction_frozen)[0,1])))
 if mae>=1.0:print('REVIEW V4 1m vs 2m target sensitivity',forest,mae,'pp')
display(pd.DataFrame(AUDIT50).round(5))
pd.DataFrame(AUDIT50).to_csv(SCALE_RUN/'08_V4_50m_sampling_sensitivity.csv',index=False)
for scale in SCALES:
 for forest in SITES:
  if scale!=50:build_scale_site(scale,forest)
print('All V4 independent multiresolution tables ready:',SCALE_RUN)


## 09 — Matched annual Sentinel-2 LOFO across spatial resolutions

**Purpose:** quantify the *resolution dependence* of the SAME annual S2 model for predicting V3.1 UAV teacher fractional cover. Primary feature family = ten annual reflectances + four indices, no temporally pooled quarters, to isolate scale from seasonality. Strict forest LOFO; capped but randomly sampled test pixels are representative of *each forest at each scale*, not matched one-to-one across scales. Each site contributes equally to summary.

**Interpretation:** different grid support changes the *target variance* itself. Compare MAE, RMSE, R² and target SD together, not just R². At 10 m, pixel footprints still represent mixtures; one spectrum is not an individual-crown signature.

In [ ]:
SCALE_SEED=20261120
MAX_TRAIN_PER_FOREST=1500
MAX_TEST_PER_FOREST=15000 # deterministic cap for rapid, unbiased within-forest sample
SCALE_FEATURES=[f'S2_{j:02d}' for j in range(1,15)]
SCALE_MODELS=['annual_reflectance_indices']
SCALE_MODEL_ARGS=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
                      min_samples_leaf=35,l2_regularization=2,early_stopping=False)
scale_metrics=[];all_out=[]
for scale in SCALES:
 samples={s:pd.read_parquet(SCALE_RUN/f'{s}_{scale}m.parquet') for s in SITES}
 for held in SITES:
  rng=np.random.default_rng(SCALE_SEED+scale*31+sum(map(ord,held)))
  target=samples[held]
  test_ids=np.arange(len(target))
  if len(test_ids)>MAX_TEST_PER_FOREST:
   test_ids=np.sort(rng.choice(test_ids,MAX_TEST_PER_FOREST,replace=False))
  training=[]
  for s in SITES:
   if s==held:continue
   other=samples[s]
   # Forest balanced training; select up to cap, stratified by abundance using training labels only.
   ids=np.arange(len(other))
   strata=np.digitize(other.target_fraction.to_numpy(),[.10,.20,.40])
   perbin=[ids[strata==i] for i in range(4)]
   pick=[]
   for b in perbin:
    if len(b):
     n=min(len(b),max(1,MAX_TRAIN_PER_FOREST//4))
     pick.extend(rng.choice(b,n,replace=False).tolist())
   remaining=MAX_TRAIN_PER_FOREST-len(pick)
   if remaining>0:
    rest=np.setdiff1d(ids,np.asarray(pick,int))
    if len(rest):pick.extend(rng.choice(rest,min(remaining,len(rest)),replace=False).tolist())
   training.append(other.iloc[np.asarray(sorted(pick),int)])
  fit_data=pd.concat(training,ignore_index=True)
  output=SCALE_RUN/f'09_LOFO_{scale}m_{held}.parquet'
  if output.is_file():
   pred_frame=pd.read_parquet(output)
   assert len(pred_frame)==len(test_ids)
   print('CACHE LOFO',scale,held,flush=True)
  else:
   model=HistGradientBoostingRegressor(random_state=SCALE_SEED,**SCALE_MODEL_ARGS)
   model.fit(fit_data[SCALE_FEATURES].to_numpy(np.float32),
             fit_data.target_fraction.to_numpy(np.float32))
   te=target.iloc[test_ids]
   pred=np.clip(model.predict(te[SCALE_FEATURES].to_numpy(np.float32)),0,1)
   pred_frame=te[['site','scale_m','row','col','target_fraction']].copy().reset_index(drop=True)
   pred_frame['prediction']=pred
   pred_frame.to_parquet(output,index=False)
   print('FIT LOFO',scale,held,'test',len(pred_frame),flush=True)
  ytrue=pred_frame.target_fraction.to_numpy(float)
  ypred=pred_frame.prediction.to_numpy(float)
  err=ypred-ytrue
  var=np.var(ytrue)
  scale_metrics.append(dict(scale_m=scale,site=held,n_test=len(pred_frame),
    n_train=len(fit_data),target_mean_pp=float(np.mean(ytrue)*100),
    target_sd_pp=float(np.std(ytrue)*100),
    MAE_pp=float(np.mean(abs(err))*100),RMSE_pp=float(np.sqrt(np.mean(err**2))*100),
    bias_pp=float(np.mean(err)*100),
    R2=float(1-np.mean(err**2)/var) if var>1e-12 else np.nan))
  all_out.append(pred_frame)
SCALE_PER=pd.DataFrame(scale_metrics)
SCALE_PER.to_csv(SCALE_RUN/'09_LOFO_per_forest_scale.csv',index=False)
# Compare equal-forest scores at each spatial support, incl. correct pooled variance.
equal=[]
for scale,g in SCALE_PER.groupby('scale_m'):
 pred=pd.concat([p for p in all_out if int(p.scale_m.iloc[0])==scale],ignore_index=True)
 means=pred.groupby('site').target_fraction.mean()
 overall=float(means.mean())
 variance=np.mean([np.mean((q.target_fraction.to_numpy()-overall)**2)
                       for _,q in pred.groupby('site')])
 mse=np.mean([np.mean((q.prediction.to_numpy()-q.target_fraction.to_numpy())**2)
                   for _,q in pred.groupby('site')])
 equal.append(dict(scale_m=scale,forests=len(g),
    equal_forest_R2=float(1-mse/variance) if variance>0 else np.nan,
    equal_forest_MAE_pp=float(g.MAE_pp.mean()),
    equal_forest_RMSE_pp=float(np.sqrt(np.mean(g.RMSE_pp**2))),
    equal_forest_target_SD_pp=float(np.sqrt(variance)*100),
    min_site_R2=float(g.R2.min()),MOS_R2=float(g[g.site.eq('MOS')].R2.iloc[0])))
SCALE_SUM=pd.DataFrame(equal).sort_values('scale_m')
SCALE_SUM.to_csv(SCALE_RUN/'09_LOFO_equal_forest_scale_summary.csv',index=False)
display(SCALE_SUM.round(3))
display(SCALE_PER.pivot(index='scale_m',columns='site',values='R2').round(3))
print('Scale prediction outputs:',SCALE_RUN)


## 10 — Scale curves and interpretation

Plot forest-equal R², absolute error and changes in reference target variance together. The **optimal scale** is not automatically the scale with highest R²: smoothing can inflate R² and suppress abundance extremes. Also inspect whether MOS responds differently from the other three forests.

This test holds raw annual input type and model family fixed, but changes predictor/target spatial support. It does **not** include Planet or seasonal imagery yet. Never claim performance at a scale that has not produced genuine observed-versus-predicted points.

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.8),layout='constrained')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_R2,'o-',label='Equal forest')
ax[0].plot(SCALE_SUM.scale_m,SCALE_SUM.MOS_R2,'s--',label='MOS only')
ax[0].set(ylabel='LOFO R²',title='Geographic transfer')
ax[0].axhline(0,ls=':',c='grey');ax[0].legend()
ax[1].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_MAE_pp,'o-')
ax[1].set(ylabel='MAE (percentage points)',title='Absolute abundance error')
ax[2].plot(SCALE_SUM.scale_m,SCALE_SUM.equal_forest_target_SD_pp,'o-')
ax[2].set(ylabel='Target SD (percentage points)',title='Target variability changes with scale')
for a in ax:
 a.set_xlabel('Aggregation scale (m)')
 a.set_xticks(SCALE_SUM.scale_m.to_numpy())
 a.grid(alpha=.15)
fig.suptitle('Native V3.1 UAV teacher + annual S2: honest multiresolution LOFO')
fig.savefig(SCALE_RUN/'10_multiresolution_LOFO_three_metrics.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
fig,axes=plt.subplots(2,2,figsize=(11,9),layout='constrained')
for axis,forest in zip(axes.flat,SITES):
 q=SCALE_PER[SCALE_PER.site.eq(forest)].sort_values('scale_m')
 axis.plot(q.scale_m,q.R2,'o-',label='R²')
 axis.axhline(0,ls='--',lw=.8,color='gray')
 axis.set(title=forest,xlabel='Aggregation scale (m)',ylabel='Held-out forest R²')
 axis.set_xticks(q.scale_m)
 axis.grid(alpha=.15)
fig.savefig(SCALE_RUN/'10_multiresolution_per_forest_R2.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Interpretation: choose a stable resolution based on MAE, forest transfer, and ecological map utility;')
print('avoid choosing scale by one inflated pooled R².')


## 11 — NICFI PlanetScope incremental information (strict 50 m matched LOFO)

The frozen 50 m reference table (built from the original 11A cached Planet inputs) contains **native PlanetScope 4.77 m B/G/R/N patch means and standard deviations**, NIR spatial texture statistics, and 10 m embedding neighbourhood means/SD. We join these columns to the *unchanged frozen 50 m target and S2* using exact `site,mosaic,row,col` keys. No raw Planet imagery is opened.

**Ablation:** annual S2 only; Planet spectral alone; Planet texture alone; annual S2 + Planet spectral; annual S2 + Planet spectral + texture; annual S2 + embedding context; all combined. Every variant in a fold uses exactly the same matched cells, forest-balanced training selection and strict held-out forest. We report per-forest R²/MAE, equal-forest R²/MAE, MOS ≥40% bias, and high-cover recall.

**Interpretation:** NICFI native pixel size is 4.77 m, but our Planet means/SD summarise roughly **100 m patches around 50 m response cells**, providing wider context. Older Planet-vs-S2 conclusions from 04B used a *Planet-only supervised model*, not identical feature families. Here we test incremental contribution of **fixed descriptive features**, not a CNN. Planet Yoko (2021) is temporally mismatched to UAV 2020.

In [ ]:
# NICFI / Planet cached-feature ablation — no raster reads.
from sklearn.ensemble import HistGradientBoostingRegressor
# V3.1 NB06 Section 00 preserved every matched NICFI/context column in the
# AUTHORITATIVE frozen reference. Do not require an extra file under CLEAN_NB06.
KEYS11=['site','mosaic','row','col']
PLANET_SPEC11=[f'PLANET_{stat}_{band}' for stat in ('mean','std') for band in ('B','G','R','N')]
PLANET_TEX11=[f'PLANET_texture_{v}' for v in ('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')]
EMB_CTX11=[f'EMB_{stat}_{j:02d}' for stat in ('mean','sd') for j in range(1,65)]
NEEDED11=PLANET_SPEC11+PLANET_TEX11+EMB_CTX11
P11=ROOT/'frozen_50m_reference.parquet'
assert P11.is_file(),('Frozen reference missing; run Section 00',P11)
J11=D.copy()
missing11=sorted(set(NEEDED11)-set(J11.columns))
assert not missing11,(
 'Frozen V3.1 reference unexpectedly lacks Planet/context features; inspect',
 missing11[:30],
 'Original fallback is under ROOT.parent / 11_spatial_ablation_50m / inputs, NOT ROOT.')
assert J11[KEYS11].notna().all().all() and not J11.duplicated(KEYS11).any()
print('Using existing frozen 50m table, including NICFI/context features:',P11)
S2_ANNUAL11=[f'RAW_ANNUAL__{i:02d}' for i in range(1,15)]
PSETS11={
 'S2_annual':S2_ANNUAL11,
 'NICFI_spectral':PLANET_SPEC11,
 'NICFI_texture':PLANET_TEX11,
 'S2_plus_NICFI_spectral':S2_ANNUAL11+PLANET_SPEC11,
 'S2_plus_NICFI_all':S2_ANNUAL11+PLANET_SPEC11+PLANET_TEX11,
 'S2_plus_embedding_context':S2_ANNUAL11+EMB_CTX11,
 'S2_NICFI_embedding_context':S2_ANNUAL11+PLANET_SPEC11+PLANET_TEX11+EMB_CTX11,
}
assert set(sum(PSETS11.values(),[])).issubset(J11.columns)
complete11=np.isfinite(J11[sorted(set(sum(PSETS11.values(),[])))].to_numpy(np.float32)).all(axis=1)
J11=J11.loc[complete11].reset_index(drop=True)
assert J11.site.nunique()==4 and len(J11)>1000,('Insufficient common Planet feature cells',J11.site.value_counts().to_dict())
print('Matched Planet + S2 sample counts, identical across seven models:')
display(J11.groupby('site').target_fraction.agg(n='size',mean='mean').round(3))
OUT11=OUT/'20_NB08_NICFI_matched_LOFO'
CFG11=dict(version='NICFI_LOFO_v4_004',source_sha=SOURCE_SHA,
 n=len(J11),keys_sha=hashlib.sha256(pd.util.hash_pandas_object(J11[KEYS11],index=False).values.tobytes()).hexdigest(),
 feature_sets=PSETS11,params=MODEL,seed=20261120,
 planet_source=dict(path=str(P11),size=P11.stat().st_size,mtime_ns=P11.stat().st_mtime_ns))
RUN11=OUT11/hashlib.sha256(json.dumps(CFG11,sort_keys=True).encode()).hexdigest()[:14]
RUN11.mkdir(parents=True,exist_ok=True)
(RUN11/'config.json').write_text(json.dumps(CFG11,indent=2))
print('NICFI output:',RUN11)


In [ ]:
# Matched LOFO fits, deterministic per-forest stratified draw; saved individually.
def draw11(d,forest,cap=1500,seed=20261120):
 rng=np.random.default_rng(seed+sum(map(ord,forest)))
 ids=np.flatnonzero(d.site.to_numpy()==forest)
 if len(ids)<=cap:return ids
 values=d.target_fraction.to_numpy(float)
 strata=np.digitize(values[ids],[.1,.2,.4])
 bins={k:ids[strata==k] for k in range(4)}
 chosen=[]
 # Reserve up to 25% for high-cover cells, if available.
 q=min(len(bins[3]),int(np.ceil(cap*.25)))
 if q:
  t=rng.choice(bins[3],q,replace=False);chosen.extend(t.tolist());bins[3]=np.setdiff1d(bins[3],t)
 while len(chosen)<cap:
  active=[k for k in bins if len(bins[k])]
  assert active
  for k in active:
   nn=min(len(bins[k]),max(1,(cap-len(chosen))//len(active)),cap-len(chosen))
   if nn:
    t=rng.choice(bins[k],nn,replace=False)
    chosen.extend(t.tolist());bins[k]=np.setdiff1d(bins[k],t)
   if len(chosen)==cap:break
 return np.asarray(sorted(chosen),int)
def fit_matched_lofo(data,featuresets,out,seed=20261120):
 yv=data.target_fraction.to_numpy(float)
 forestv=data.site.to_numpy()
 per=[];preds=[]
 for held in SITES:
  test=np.flatnonzero(forestv==held)
  train_ids=np.concatenate([draw11(data,s,seed=seed) for s in SITES if s!=held])
  assert len(np.intersect1d(test,train_ids))==0
  frame=data.iloc[test][KEYS11+['target_fraction']].reset_index(drop=True).copy()
  for name,cols in featuresets.items():
   file=out/f'{held}_{name}.npz'
   if file.is_file():
    with np.load(file,allow_pickle=False) as z:
     assert np.array_equal(z['test'],test)
     yp=z['pred'].astype(float)
    print('CACHE',held,name,flush=True)
   else:
    h=HistGradientBoostingRegressor(random_state=seed,**MODEL)
    h.fit(data.iloc[train_ids][cols].to_numpy(np.float32),yv[train_ids])
    yp=np.clip(h.predict(data.iloc[test][cols].to_numpy(np.float32)),0,1)
    tmp=file.with_suffix('.partial')
    with open(tmp,'wb') as fd:np.savez_compressed(fd,test=test,pred=yp.astype(np.float32))
    os.replace(tmp,file)
    print('FIT',held,name,flush=True)
   frame[name]=yp
   err=yp-yv[test];variance=np.var(yv[test])
   hi=yv[test]>=.4
   per.append(dict(site=held,model=name,n=len(test),n_train=len(train_ids),
     MAE_pp=100*np.mean(abs(err)),RMSE_pp=100*np.sqrt(np.mean(err**2)),
     MSE=np.mean(err**2),bias_pp=100*np.mean(err),
     R2=1-np.mean(err**2)/variance if variance>0 else np.nan,
     high_n=int(hi.sum()),high_bias_pp=100*np.mean(err[hi]) if hi.any() else np.nan,
     high_recall=float(np.mean(yp[hi]>=.4)) if hi.any() else np.nan))
  preds.append(frame)
 PRED=pd.concat(preds,ignore_index=True)
 PER=pd.DataFrame(per)
 mu=PRED.groupby('site').target_fraction.mean().mean()
 variance=np.mean([np.mean((g.target_fraction.to_numpy()-mu)**2) for _,g in PRED.groupby('site')])
 SUM=[]
 for name in featuresets:
  g=PER[PER.model.eq(name)]
  SUM.append(dict(model=name,n=len(PRED),n_forests=4,
   equal_forest_R2=1-g.MSE.mean()/variance,
   equal_forest_MAE_pp=g.MAE_pp.mean(),
   equal_forest_RMSE_pp=np.sqrt(np.mean(g.RMSE_pp**2)),
   MOS_R2=float(g[g.site.eq('MOS')].R2.iloc[0]),
   MOS_high_bias_pp=float(g[g.site.eq('MOS')].high_bias_pp.iloc[0]),
   MOS_high_recall=float(g[g.site.eq('MOS')].high_recall.iloc[0])))
 SUM=pd.DataFrame(SUM)
 PER.to_csv(out/'LOFO_per_forest.csv',index=False)
 SUM.to_csv(out/'LOFO_equal_forest_summary.csv',index=False)
 PRED.to_parquet(out/'LOFO_matched_predictions.parquet',index=False)
 return PRED,PER,SUM
PRED11,PER11,SUM11=fit_matched_lofo(J11,PSETS11,RUN11)
display(SUM11.round(3))
print('Per forest:');display(PER11.pivot(index='site',columns='model',values='R2').round(3))
fig,axes=plt.subplots(1,2,figsize=(14,4.8),layout='constrained')
axes[0].bar(SUM11.model,SUM11.equal_forest_R2)
axes[0].set(title='Incremental NICFI / context transfer',ylabel='Equal-forest R²')
axes[1].bar(SUM11.model,SUM11.MOS_high_bias_pp)
axes[1].axhline(0,color='black',ls='--',lw=.8)
axes[1].set(title='MOS high-cover calibration (observed ≥40%)',ylabel='Prediction bias (pp)')
for a in axes:a.tick_params(axis='x',labelrotation=55);a.grid(axis='y',alpha=.15)
fig.savefig(RUN11/'11_NICFI_incremental_performance.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)


## 12 — Control for independently sourced forest height, tree cover and disturbance

The **existing cached `07A_joined_UAV_and_structure.parquet`** was sampled at a *random, forest-stratified subset* of the frozen 50 m reference centres. It includes independent products:

- **ETH global canopy height 2020** (10 m modelled height), local mean and SD;
- **JRC Tropical Moist Forest status December 2020**: undisturbed, degraded and regrowth neighbourhood fractions;
- **Copernicus 2019 tree-cover fraction** (100 m native product; not secretly 10 m).

These are imperfect **forest-condition controls**, not measured ground-truth canopy height or causal covariates. The 2020 layers can postdate Yoko's 2020 UAV acquisition or precede 2023/24 UAV dates; this is a temporal-confounding limitation.

We match the exact archived 07A sample keys and use **the same held-out observations** for (i) structure only, (ii) annual S2 only, (iii) annual S2 + structure. Incremental test: improvement in LOFO R²/MAE over structure-only indicates satellite signal beyond these *measured proxies*, not necessarily beyond true forest age/structure. If both predict similarly, ecological confounding becomes more plausible. The same complete-case sample is used across variants; no site-specific models or held-out target-informed feature selection. Results and coverage counts are always displayed.

In [ ]:
REFLECTANCE=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
INDICES=[f'RAW_ANNUAL__{i:02d}' for i in range(11,15)]
STRUCTURE_PATH12=ROOT/'07_structure_products_screen/07A_joined_UAV_and_structure.parquet'
assert STRUCTURE_PATH12.is_file(),('Missing cached independent forest-condition sample',STRUCTURE_PATH12)
T12=pd.read_parquet(STRUCTURE_PATH12)
STRUCT_COLS12=['height_local_mean_m','height_local_sd_m','height_model_sd_local_m',
 'tmf_undisturbed_local_frac','tmf_degraded_local_frac','tmf_regrowth_local_frac',
 'cop_2019_tree_fraction_pct']
assert set(KEYS11+STRUCT_COLS12).issubset(T12.columns),sorted(set(KEYS11+STRUCT_COLS12)-set(T12.columns))
assert not T12.duplicated(KEYS11).any()
ST12=D.merge(T12[KEYS11+STRUCT_COLS12],how='inner',on=KEYS11,validate='one_to_one')
assert len(ST12)>500 and ST12.site.nunique()==4,('Expected 4-forest independent control subset',len(ST12))
coverage12=ST12.groupby('site')[STRUCT_COLS12].agg(lambda z:100*z.notna().mean())
print('Available independent forest-condition control data, % by forest:')
display(coverage12.round(1))
ST12=ST12.dropna(subset=STRUCT_COLS12+REFLECTANCE+INDICES).reset_index(drop=True)
assert ST12.site.nunique()==4 and len(ST12)>500,('Insufficient matched structure sample',ST12.site.value_counts().to_dict())
PSETS12={
 'forest_conditions_only':STRUCT_COLS12,
 'S2_annual_only':REFLECTANCE+INDICES,
 'forest_conditions_plus_S2':STRUCT_COLS12+REFLECTANCE+INDICES}
OUT12=OUT/'21_NB08_forest_condition_controls'
CFG12=dict(version='forest_structure_07A_LOFO_v4_004',source_sha=SOURCE_SHA,
  key_sha=hashlib.sha256(pd.util.hash_pandas_object(ST12[KEYS11],index=False).values.tobytes()).hexdigest(),
  n=len(ST12),features=PSETS12,
  sample_source=dict(path=str(STRUCTURE_PATH12),size=STRUCTURE_PATH12.stat().st_size,
                     mtime_ns=STRUCTURE_PATH12.stat().st_mtime_ns),
  sampling='Independent 2019-2020 forest structure proxies at 50m cell centres; native 100m tree cover')
RUN12=OUT12/hashlib.sha256(json.dumps(CFG12,sort_keys=True).encode()).hexdigest()[:14]
RUN12.mkdir(parents=True,exist_ok=True)
(RUN12/'config.json').write_text(json.dumps(CFG12,indent=2))
print('Matched control data by forest:');display(ST12.groupby('site').size().rename('n'))
PRED12,PER12,SUM12=fit_matched_lofo(ST12,PSETS12,RUN12)
display(SUM12.round(3))
display(PER12.pivot(index='site',columns='model',values='R2').round(3))
fig,ax=plt.subplots(1,2,figsize=(12,4.8),layout='constrained')
ax[0].bar(SUM12.model,SUM12.equal_forest_R2);ax[0].set(ylabel='Equal-forest R²',title='Independent structure control')
ax[1].bar(SUM12.model,SUM12.equal_forest_MAE_pp);ax[1].set(ylabel='MAE (pp)',title='Forest-equal absolute error')
for a in ax:a.tick_params(axis='x',labelrotation=18);a.grid(axis='y',alpha=.15)
fig.savefig(RUN12/'12_structure_control_incremental_LOFO.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Caution: temporal and native-resolution mismatch; a proxy-only model is not evidence of causal identification.')


## 13 — Separate UAV DSM sensitivity: *relative surface shape*, not canopy height

An existing archived 16B DSM analysis measures `roughness_sd_m` and `relative_surface_m` in and around frozen 50 m cells. These variables depend on terrain, sensor geometry and reconstruction quality. Crucially, **DSM ≠ canopy-height model (CHM)**; no digital terrain model is used. The 16B DSM covers only **MOS_1_2**, not MOS_5_6, so sample availability is geographically uneven.

**Purpose:** exploratory matched LOFO comparison of annual S2 versus S2+DSM roughness on the valid 16B subset. It may fail the 4-forest support requirement; in that case this block stops with a clear diagnostic rather than substituting missing values or misrepresenting the limited sample. Do not combine this subset's metrics with the primary NICFI comparison.

In [ ]:
DSM_PATH13=ROOT/'16_DSM_structure/16B_reference_cells/16B_frozen_cells_DSM_metrics.parquet'
assert DSM_PATH13.is_file(),('Cached DSM diagnostic missing',DSM_PATH13)
DSM13=pd.read_parquet(DSM_PATH13)
DSM_COLS13=['roughness_sd_m','relative_surface_m','ring_residual_sd_m','ring_slope_m_per_m']
assert set(KEYS11+DSM_COLS13).issubset(DSM13.columns)
assert not DSM13.duplicated(KEYS11).any()
DS13=D.merge(DSM13[KEYS11+DSM_COLS13],on=KEYS11,how='inner',validate='one_to_one')
DS13=DS13.replace([np.inf,-np.inf],np.nan).dropna(subset=DSM_COLS13+REFLECTANCE+INDICES).reset_index(drop=True)
print('Valid DSM-matched sample by forest:');display(DS13.groupby('site').size().reindex(SITES,fill_value=0))
if set(DS13.site.unique())!=set(SITES) or DS13.groupby('site').size().min()<30:
 print('DSM subset is not suitable for a full 4-forest LOFO claim. Descriptive association only; no model fitted.')
else:
 PSETS13={'S2_annual':REFLECTANCE+INDICES,
   'DSM_relative_surface_only':DSM_COLS13,
   'S2_plus_DSM_surface':REFLECTANCE+INDICES+DSM_COLS13}
 OUT13=OUT/'22_NB08_DSM_exploratory_control'
 CFG13=dict(version='DSM_exploratory_v4_004',source_sha=SOURCE_SHA,
   keys_sha=hashlib.sha256(pd.util.hash_pandas_object(DS13[KEYS11],index=False).values.tobytes()).hexdigest(),
   features=PSETS13,n=len(DS13),dsm_mtime=DSM_PATH13.stat().st_mtime_ns)
 RUN13=OUT13/hashlib.sha256(json.dumps(CFG13,sort_keys=True).encode()).hexdigest()[:14]
 RUN13.mkdir(parents=True,exist_ok=True)
 (RUN13/'config.json').write_text(json.dumps(CFG13,indent=2))
 PRED13,PER13,SUM13=fit_matched_lofo(DS13,PSETS13,RUN13)
 display(SUM13.round(3))
 fig,ax=plt.subplots(figsize=(8,4.5),layout='constrained')
 ax.bar(SUM13.model,SUM13.equal_forest_R2)
 ax.tick_params(axis='x',rotation=18);ax.set(ylabel='Equal-forest R²',title='DSM relative surface / roughness (exploratory)')
 fig.savefig(RUN13/'13_DSM_matched_LOFO.png',dpi=180)
 plt.show();plt.close(fig)
print('NB08 v4_004 complete. Original validated experiments and output folders were not modified.')


## 14 — Pre-specified matched spectral contrast within forests

This is a **conditional association test**, not a further supervised abundance model. On the existing *random* 07A reference-cell sample, compare high Musanga (UAV fraction ≥40%) with low Musanga (≤5%). Restrict comparisons **within the same forest**. The forest-condition control dimensions are (i) ETH 2020 10 m modelled canopy height and local height variability, (ii) JRC 2020 proportions of undisturbed/degraded/regrowth tropical forest, and (iii) Copernicus 2019 tree-cover fraction at native 100 m.

Choose matches **blind to the Sentinel-2 reflectance and vegetation indices**. Greedy, one-to-one no-replacement matching, with strict clinical-like calipers and ≥150 m separation between matched cell centres. Selection is repeated with a second, looser matching specification as an explicitly labelled sensitivity test. Report treatment/control attrition, actual covariate imbalance, and why a site is not estimable. **Do not silently pool Yangambi**: it has few/no ≥40% positive cells.

Primary pre-specified response is *B8 NIR reflectance*, as a directly physical band. Red-edge B6/B7, B8A, SWIR and NDVI are secondary. Assess the **shape** of the paired spectral contrast, not only significance. The matching conditions are imperfect proxies, may be partially satellite-derived, and cannot remove all ecological confounding.

In [ ]:
# Section 14: deterministic, label-blind spectral-distance matching.
from scipy.spatial.distance import cdist
from scipy.spatial import cKDTree
from sklearn.preprocessing import RobustScaler
from scipy.stats import binomtest
SEED14=20261120
RNG14=np.random.default_rng(SEED14)
KEY14=['site','mosaic','row','col']
CONTROLS14=['height_local_mean_m','height_local_sd_m',
 'tmf_undisturbed_local_frac','tmf_degraded_local_frac','tmf_regrowth_local_frac',
 'cop_2019_tree_fraction_pct']
SRC14=ROOT/'07_structure_products_screen/07A_joined_UAV_and_structure.parquet'
assert SRC14.exists(),('Original 07A forest-condition cache missing',SRC14)
control14=pd.read_parquet(SRC14)
assert not control14.duplicated(KEY14).any()
assert set(KEY14+CONTROLS14).issubset(control14.columns)
D14=D.merge(control14[KEY14+CONTROLS14],on=KEY14,how='inner',validate='one_to_one')
D14=D14.replace([np.inf,-np.inf],np.nan).dropna(subset=CONTROLS14+['x','y']+
     [f'RAW_ANNUAL__{i:02d}' for i in range(1,15)]).reset_index(drop=True)
assert D14.site.nunique()==4
BANDS14=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
WAVELENGTH14=[492,560,665,833,704,740,783,865,1610,2190]
SPECTRA14=[f'RAW_ANNUAL__{i:02d}' for i in range(1,11)]
assert len(SPECTRA14)==10
spec14={b:c for b,c in zip(BANDS14,SPECTRA14)}
# Tests use covariate-only distances. Label defines cases and controls, never distance.
SPEC14={
 'primary':dict(height_max=8.0,tree_max=20.0,regrowth_max=.35,scaled_distance_max=3.0,min_space_m=150.),
 'sensitivity':dict(height_max=12.0,tree_max=30.0,regrowth_max=.50,scaled_distance_max=4.5,min_space_m=150.)}
OUT14=OUT/'23_NB08_matched_ecological_spectral'
CFG14=dict(version='conditional_spectral_v4_005',seed=SEED14,source_sha=SOURCE_SHA,
 match_specs=SPEC14,control_variables=CONTROLS14,
 thresholds={'high':.40,'low':.05},primary_band='B8',unit='reflectance',
 structure_sample=str(SRC14),mtime_ns=SRC14.stat().st_mtime_ns)
RUN14=OUT14/hashlib.sha256(json.dumps(CFG14,sort_keys=True).encode()).hexdigest()[:14]
RUN14.mkdir(parents=True,exist_ok=True)
(RUN14/'config.json').write_text(json.dumps(CFG14,indent=2))
print('Reference availability and abundance prevalence:')
display(D14.assign(group=np.select([D14.target_fraction>=.4,D14.target_fraction<=.05],
                                    ['high','low'],default='middle')).groupby(['site','group']).size().unstack(fill_value=0))
def match14(part,options,seed):
 high=part[part.target_fraction>=.40].copy()
 low=part[part.target_fraction<=.05].copy()
 if high.empty or low.empty:
  return [],dict(n_high=len(high),n_low=len(low),n_pairs=0,reason='No high and/or low observed cells')
 # Scale controls using this site's pooled unlabeled covariate distribution only.
 scalar=RobustScaler(quantile_range=(25,75)).fit(part[CONTROLS14])
 zhigh=scalar.transform(high[CONTROLS14])
 zlow=scalar.transform(low[CONTROLS14])
 delta=cdist(zhigh,zlow,'euclidean')
 height=np.abs(high.height_local_mean_m.to_numpy()[:,None]-low.height_local_mean_m.to_numpy()[None,:])
 tree=np.abs(high.cop_2019_tree_fraction_pct.to_numpy()[:,None]-low.cop_2019_tree_fraction_pct.to_numpy()[None,:])
 reg=np.abs(high.tmf_regrowth_local_frac.to_numpy()[:,None]-low.tmf_regrowth_local_frac.to_numpy()[None,:])
 spatial=cdist(high[['x','y']],low[['x','y']])
 eligible=(height<=options['height_max'])&(tree<=options['tree_max'])&(
  reg<=options['regrowth_max'])&(spatial>=options['min_space_m'])&(
  delta<=options['scaled_distance_max'])
 # Prioritize most constrained high cells, then the nearest low cell with no reuse.
 count=eligible.sum(1)
 order=np.lexsort((np.arange(len(high)),count))
 assigned=set();matched=[]
 for ix in order:
  options_low=np.flatnonzero(eligible[ix])
  if len(options_low)==0:continue
  free=[j for j in options_low if j not in assigned]
  if not free:continue
  j=min(free,key=lambda k:(delta[ix,k],k))
  assigned.add(j)
  a=high.iloc[ix];b=low.iloc[j]
  rec=dict(site=str(a.site),high_mosaic=str(a.mosaic),low_mosaic=str(b.mosaic),
    high_row=int(a.row),high_col=int(a.col),low_row=int(b.row),low_col=int(b.col),
    high_x=float(a.x),high_y=float(a.y),low_x=float(b.x),low_y=float(b.y),
    high_cover=float(a.target_fraction),low_cover=float(b.target_fraction),
    match_distance=float(delta[ix,j]),match_spatial_m=float(spatial[ix,j]))
  for feature in CONTROLS14:
   rec['high_'+feature]=float(a[feature]);rec['low_'+feature]=float(b[feature])
  for band,col in spec14.items():
   rec['high_'+band]=float(a[col]);rec['low_'+band]=float(b[col])
   rec['delta_'+band]=float(a[col]-b[col])
  for c in ['RAW_ANNUAL__11','RAW_ANNUAL__12','RAW_ANNUAL__13','RAW_ANNUAL__14']:
   rec['delta_'+c]=float(a[c]-b[c])
  # Blocks are spatial units, not one block per matched cell; high-side block
  # defines the bootstrap cluster. Re-used low-side locations are prohibited.
  rec['block500']=f"{a.site}_{int(np.floor(a.x/500))}_{int(np.floor(a.y/500))}"
  matched.append(rec)
 summary=dict(n_high=len(high),n_low=len(low),n_pairs=len(matched),
   eligible_high=int(np.any(eligible,axis=1).sum()),
   median_distance=float(np.median([q['match_distance'] for q in matched])) if matched else np.nan,
   reason='ok' if matched else 'No matches inside all prespecified calipers')
 return matched,summary
PAIRS14={};ATTRITION14=[];BALANCE14=[]
for setting,opts in SPEC14.items():
 matches=[]
 for forest in SITES:
  subs=D14[D14.site.eq(forest)]
  pairs,details=match14(subs,opts,SEED14)
  ATTRITION14.append(dict(specification=setting,site=forest,**details))
  matches.extend(pairs)
  if pairs:
   mp=pd.DataFrame(pairs)
   pool=subs[CONTROLS14]
   # SMD = paired mean difference / within-site source SD; this does NOT prove balance.
   for col in CONTROLS14:
    sd=float(pool[col].std())
    diff=mp['high_'+col]-mp['low_'+col]
    BALANCE14.append(dict(specification=setting,site=forest,variable=col,
      mean_high=float(mp['high_'+col].mean()),mean_low=float(mp['low_'+col].mean()),
      paired_mean_difference=float(diff.mean()),
      standardized_difference=float(diff.mean()/sd) if sd>0 else np.nan))
 PAIRS14[setting]=pd.DataFrame(matches)
 PAIRS14[setting].to_csv(RUN14/f'14_matched_pairs_{setting}.csv',index=False)
pd.DataFrame(ATTRITION14).to_csv(RUN14/'14_matching_attrition.csv',index=False)
pd.DataFrame(BALANCE14).to_csv(RUN14/'14_matching_covariate_balance.csv',index=False)
print('Matching attrition — do not interpret a site with zero overlap:')
display(pd.DataFrame(ATTRITION14).round(3))
print('Maximum absolute standardized covariate difference (ideally <0.1):')
balance14=pd.DataFrame(BALANCE14)
if len(balance14):display(balance14.groupby(['specification','site']).standardized_difference.apply(
 lambda z:float(z.abs().max())).round(3))


## 15 — Paired spectral contrasts with spatially clustered confidence intervals

Pre-specified main test: median and mean matched **high-minus-low B8 reflectance**. Re-sample spatial **500 m blocks** of matched high-cover cells, rather than treating every neighbouring 50 m cell as an independent replicate. The confidence intervals are **conditional on the greedy matching solution** (they do not incorporate uncertainty in matching or the underlying UAV segmentation). A spatial-block sign-flip randomization check is reported as an approximate sensitivity analysis, **not** a definitive causal p-value.

Estimate individual forests separately and only summarize matched forests. The all-site mean is forest-equal; with only three potential positive forests, forest-level generalization remains limited. Plot the entire spectral profile and mark the NIR/red-edge versus visible. Apply **Benjamini–Hochberg FDR to secondary spectral-band tests**; primary B8 is declared in advance.

In [ ]:
from scipy.stats import binomtest
BOOT14=2000
def bootstrap_blocks14(df,col,seed):
 vals=df[col].to_numpy(float)
 keys=df.block500.to_numpy()
 _,label=np.unique(keys,return_inverse=True)
 groups=[np.flatnonzero(label==k) for k in range(label.max()+1)]
 block_sums=np.array([vals[g].sum() for g in groups])
 block_ns=np.array([len(g) for g in groups])
 raw=float(vals.mean())
 if len(groups)<3:return raw,np.nan,np.nan,np.nan,len(groups)
 rng=np.random.default_rng(seed)
 draws=rng.integers(0,len(groups),size=(BOOT14,len(groups)))
 sums=block_sums[draws].sum(1)
 counts=block_ns[draws].sum(1)
 dist=sums/counts
 lo,hi=np.quantile(dist,[.025,.975])
 # Approximate block sign randomization; valid only under stronger exchangeability.
 signs=rng.choice([-1.,1.],size=(BOOT14,len(groups)))
 null=(signs*block_sums).sum(1)/block_ns.sum()
 p=(1+np.sum(np.abs(null)>=abs(raw)))/(BOOT14+1)
 return raw,float(lo),float(hi),float(p),len(groups)
ROWS15=[]
for name,pairs in PAIRS14.items():
 if pairs.empty:continue
 for forest,g in list(pairs.groupby('site'))+[('forest_equal',pairs)]:
  # For forest-equal summary first average within each site/replicate,
  # rather than simply pooling MOS-heavy matched pairs.
  if forest=='forest_equal':
   continue
  for band in BANDS14:
   value,lo,hi,p,nblocks=bootstrap_blocks14(g,'delta_'+band,SEED14+sum(map(ord,forest))+BANDS14.index(band))
   ROWS15.append(dict(specification=name,site=forest,band=band,n_pairs=len(g),
    n_spatial_blocks=nblocks,delta_reflectance=value,ci_low=lo,ci_high=hi,
    block_signflip_p=p,primary_test=band=='B8'))
STATS15=pd.DataFrame(ROWS15)
assert not STATS15.empty,('No scientifically valid within-forest high-low matched pairs at current calipers',ATTRITION14)
# Forest-equal mean with resampling of 500m blocks within each site.
for name,pairs in PAIRS14.items():
 if pairs.empty:continue
 forests=[s for s,g in pairs.groupby('site') if len(g)>=5 and g.block500.nunique()>=3]
 if not forests:
  print('No site with >=5 pairs and >=3 independent 500m blocks under',name)
  continue
 for band in BANDS14:
  per_site=[pairs[pairs.site.eq(f)] for f in forests]
  mean=float(np.mean([g['delta_'+band].mean() for g in per_site]))
  rng=np.random.default_rng(SEED14+10000+BANDS14.index(band)+(name=='sensitivity'))
  arr=np.zeros(BOOT14)
  for gg in per_site:
   gr=[v['delta_'+band].to_numpy(float) for _,v in gg.groupby('block500')]
   blocksum=np.array([v.sum() for v in gr]);blocksize=np.array([len(v) for v in gr])
   picks=rng.integers(0,len(gr),size=(BOOT14,len(gr)))
   arr+=(blocksum[picks].sum(axis=1)/blocksize[picks].sum(axis=1))/len(per_site)
  ci=np.quantile(arr,[.025,.975])
  STATS15.loc[len(STATS15)]=dict(specification=name,site='forest_equal',band=band,
      n_pairs=sum(map(len,per_site)),n_spatial_blocks=sum(g.block500.nunique() for g in per_site),
      delta_reflectance=mean,ci_low=ci[0],ci_high=ci[1],
      block_signflip_p=np.nan,primary_test=band=='B8')
# Exploratory bands: BH FDR within site and matching specification.
# The prespecified primary B8 contrast is excluded from this multiple-test family.
STATS15['secondary_FDR_q']=np.nan
secondary=STATS15[(STATS15.band!='B8')&(STATS15.site!='forest_equal')]
for (design,forest),idx in secondary.groupby(['specification','site']).groups.items():
 p=STATS15.loc[idx,'block_signflip_p'].to_numpy(float)
 valid=np.flatnonzero(np.isfinite(p))
 if len(valid):
  order=valid[np.argsort(p[valid])]
  m=len(valid)
  qsort=np.minimum.accumulate((p[order]*m/np.arange(1,m+1))[::-1])[::-1]
  STATS15.loc[STATS15.loc[idx].index[order],'secondary_FDR_q']=np.minimum(1,qsort)
STATS15.to_csv(RUN14/'15_paired_spectral_bootstrap_summary.csv',index=False)
display(STATS15[STATS15.band.eq('B8')].round(5))
# Figure 1: matched paired physical spectral signature by site, plotted by real wavelength.
fig,axes=plt.subplots(2,2,figsize=(13,9),layout='constrained')
for ax,forest in zip(axes.flat,SITES):
 part=STATS15[(STATS15.specification=='primary')&(STATS15.site==forest)]
 if len(part)==0:
  ax.text(.5,.5,'No suitable high–low matches',ha='center',va='center',transform=ax.transAxes)
  ax.set_title(forest+' — no common support')
  continue
 part=part.set_index('band').loc[BANDS14].copy()
 ww=np.asarray(WAVELENGTH14)
 order=np.argsort(ww)
 for j in order:
  v=part.iloc[j]
  if pd.notna(v.ci_low):
   ax.vlines(ww[j],v.ci_low,v.ci_high,color='steelblue',alpha=.7,lw=3)
 ax.plot(ww[order],part.delta_reflectance.to_numpy()[order],'o-',color='black')
 ax.axhline(0,color='gray',ls='--',lw=.8)
 ax.set(title=f'{forest} | matched n={int(part.n_pairs.iloc[0])}',xlabel='Wavelength (nm)',
        ylabel='High − low reflectance')
 ax.grid(alpha=.12)
fig.suptitle('Within-forest, forest-condition-matched Musanga reflectance associations')
fig.savefig(RUN14/'15_matched_spectral_difference_by_forest.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
# Figure 2: compare primary versus sensitivity design at B8 across forests.
main=STATS15[(STATS15.band=='B8')&(STATS15.site!='forest_equal')]
fig,ax=plt.subplots(figsize=(9,5),layout='constrained')
names=list(SITES)
for i,design in enumerate(SPEC14):
 part=main[main.specification.eq(design)].set_index('site')
 x=np.arange(len(names))+(i-.5)*.18
 for j,f in enumerate(names):
  if f not in part.index:continue
  z=part.loc[f]
  ax.plot(x[j],z.delta_reflectance,'o',label=design if j==0 else None)
  if np.isfinite(z.ci_low):
   ax.vlines(x[j],z.ci_low,z.ci_high,color='black',lw=1.5)
ax.axhline(0,ls='--',color='gray')
ax.set_xticks(np.arange(len(names)),names)
ax.set(ylabel='Matched high − low B8 reflectance',title='Pre-specified B8 contrast and matching sensitivity')
ax.legend()
fig.savefig(RUN14/'15_B8_matched_primary_vs_sensitivity.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)
print('Block bootstrap CIs reflect spatial clustering, not reference or model measurement uncertainty.')


## 16 — Identification-quality report and manuscript-ready synthesis

The main results should only be called **conditional spectral associations** if (a) enough matched pairs survive both strict and sensitivity calipers, (b) covariate standardized mean differences are small after matching, (c) the B8 sign is consistent across estimable forests, and (d) spatial-block confidence intervals do not contradict the interpretation. A negative or inconclusive result is scientifically valuable.

Extra diagnostics: plot covariate-balance Love plot; show matched-condition distributions; write a summary table distinguishing *predictive skill*, *conditional spectral association* and *species uniqueness*. Do not infer species specificity in the absence of comparable data for other pioneer species. Avoid causal terminology such as 'Musanga causes greater NIR' from observational matching.

In [ ]:
# Condensed diagnostics + researcher-facing report; no further model fits.
bal=balance14[balance14.specification.eq('primary')]
if not bal.empty:
 pivot=bal.pivot(index='variable',columns='site',values='standardized_difference')
 fig,ax=plt.subplots(figsize=(10,5),layout='constrained')
 for j,forest in enumerate(SITES):
  if forest not in pivot:continue
  vals=pivot[forest].to_numpy(float)
  ax.scatter(vals,np.arange(len(pivot))+(j-1.5)*.12,label=forest,s=35)
 ax.axvline(-.1,color='gray',ls='--',lw=1)
 ax.axvline(.1,color='gray',ls='--',lw=1)
 ax.axvline(0,color='gray',lw=.8)
 ax.set_yticks(np.arange(len(pivot)),pivot.index)
 ax.set(xlabel='Postmatch standardized mean difference',title='Ecological match quality (target |SMD| < 0.1)')
 ax.legend()
 fig.savefig(RUN14/'16_ecological_matching_balance.png',dpi=180,bbox_inches='tight')
 plt.show();plt.close(fig)
report=[]
report.append('NB08 v4_005 | conditional Musanga satellite spectral association')
report.append('Source: frozen V3.1 50 m teacher and previously cached, independent-product 07A sample')
report.append('Prespecified response: B8 NIR reflectance, high cover >=40% minus low cover <=5%')
report.append('Matching: within forest, no replacement, canopy-height and JRC/Copernicus controls, minimum 150m pair separation')
report.append('Primary and sensitivity calipers both declared before inspection of results')
report.append('IMPORTANT: Matching cannot establish uniqueness relative to other pioneer tree species')
for z in ATTRITION14:
 report.append(f"{z['specification']} {z['site']}: high={z['n_high']} low={z['n_low']} matched={z['n_pairs']} | {z.get('reason','')}")
for z in STATS15[(STATS15.band=='B8')].itertuples():
 report.append(f"{z.specification} {z.site}: B8 mean delta={z.delta_reflectance:.5f} "
   f"95% block CI=[{z.ci_low:.5f},{z.ci_high:.5f}], n={z.n_pairs}, blocks={z.n_spatial_blocks}")
report.append('Inferences conditional on matched sample, control quality, 2020/2019 temporal mismatches, and segmentation reference')
(RUN14/'16_scientific_interpretation.txt').write_text('\n'.join(report)+'\n')
print('\n'.join(report[-20:]))
print('Saved scientific audit:',RUN14)


## 17 — Balance-qualified spectral matching (NB08 final extension)

Re-run high (≥40%) versus low (≤5%) comparisons on the same archived 07A sample. **No satellite spectral variable enters matching.** Optimize one-to-one matching within forest, enforce 150 m separation, and assess *all six* forest-condition controls. Report treatment attrition, 500 m cluster count, and standardized mean differences. Only describe spectral contrasts as balance-qualified when every absolute standardized mean difference ≤0.10, ≥15 pairs, and ≥5 spatial blocks. Nonqualifying analyses are explicitly inconclusive. Yangambi is not imputed. This is an observational test, not species-specific or causal identification.


In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
SEED17=20261120
SPECS17={'strict':dict(height=5.,tree=10.,regrowth=.20,dist=2.0),
         'sensitivity':dict(height=8.,tree=15.,regrowth=.30,dist=3.0)}
CONT17=CONTROLS14
OUT17=OUT/'24_NB08_final_balance_validation'
OUT17.mkdir(parents=True,exist_ok=True)
MIN_PAIRS17=15;MIN_BLOCKS17=5;SMD_LIMIT17=.10
def match17(forest,spec):
 d=D14[D14.site.eq(forest)].reset_index(drop=True)
 h=d[d.target_fraction>=.40].reset_index(drop=True)
 l=d[d.target_fraction<=.05].reset_index(drop=True)
 info=dict(site=forest,specification=spec,n_high=len(h),n_low=len(l),n_initial=0,n_retained=0,
           n_blocks=0,max_abs_smd=np.nan,accepted=False,reason='')
 if len(h)==0 or len(l)==0:
  info['reason']='No high/low common abundance classes';return [],info,[]
 source=d[CONT17].to_numpy(float);sd=source.std(axis=0,ddof=1)
 active=sd>1e-10;sd=np.where(active,sd,1.)
 if not active.all():
  info['reason']='One or more constant control covariates; overlap not established';return [],info,[]
 zh=h[CONT17].to_numpy(float)/sd;zl=l[CONT17].to_numpy(float)/sd
 dist=cdist(zh,zl)
 rule=SPECS17[spec]
 spatial=cdist(h[['x','y']],l[['x','y']])
 eligible=(np.abs(h.height_local_mean_m.to_numpy()[:,None]-l.height_local_mean_m.to_numpy()[None,:])<=rule['height'])
 eligible&=(np.abs(h.cop_2019_tree_fraction_pct.to_numpy()[:,None]-l.cop_2019_tree_fraction_pct.to_numpy()[None,:])<=rule['tree'])
 eligible&=(np.abs(h.tmf_regrowth_local_frac.to_numpy()[:,None]-l.tmf_regrowth_local_frac.to_numpy()[None,:])<=rule['regrowth'])
 eligible&=(dist<=rule['dist'])&(spatial>=150.)
 # Dummy columns allow high cells to remain unmatched.
 penalty=10000.
 cost=np.concatenate((np.where(eligible,dist,penalty*2),np.full((len(h),len(h)),penalty)),axis=1)
 ii,jj=linear_sum_assignment(cost)
 pairs=[(int(i),int(j)) for i,j in zip(ii,jj) if j<len(l) and eligible[i,j]]
 info['n_initial']=len(pairs)
 # Outcome-blind balance trimming: remove the pair most contributing to the most imbalanced mean.
 # Never use spectral measurements, and stop before the minimum required sample size.
 while len(pairs)>MIN_PAIRS17:
  differences=np.array([zh[i]-zl[j] for i,j in pairs])
  smd=differences.mean(axis=0)
  if np.max(np.abs(smd))<=SMD_LIMIT17:break
  worst=int(np.argmax(np.abs(smd)))
  contributions=np.sign(smd[worst])*differences[:,worst]
  pairs.pop(int(np.argmax(contributions)))
 if pairs:
  diffs=np.array([zh[i]-zl[j] for i,j in pairs])
  smd=diffs.mean(axis=0)
  info['max_abs_smd']=float(np.max(np.abs(smd)))
 else:smd=np.full(len(CONT17),np.nan)
 rows=[];blocks=set()
 for ix,(i,j) in enumerate(pairs):
  a=h.iloc[i];b=l.iloc[j]
  block=f"{forest}_{int(a.x//500)}_{int(a.y//500)}";blocks.add(block)
  record=dict(site=forest,specification=spec,pair=ix,block500=block,
              high_cover=float(a.target_fraction),low_cover=float(b.target_fraction),
              high_x=float(a.x),high_y=float(a.y),low_x=float(b.x),low_y=float(b.y))
  for feature in CONT17:
   record['high_'+feature]=float(a[feature]);record['low_'+feature]=float(b[feature])
  for band,col in spec14.items():record['delta_'+band]=float(a[col]-b[col])
  rows.append(record)
 info['n_retained']=len(rows);info['n_blocks']=len(blocks)
 info['accepted']=bool(len(rows)>=MIN_PAIRS17 and len(blocks)>=MIN_BLOCKS17 and np.all(np.abs(smd)<=SMD_LIMIT17))
 info['reason']='Balance and support pass' if info['accepted'] else 'Failed balance and/or support; spectral attribution inconclusive'
 balance=[dict(site=forest,specification=spec,control=feature,smd=float(v),passes=bool(abs(v)<=SMD_LIMIT17)) for feature,v in zip(CONT17,smd)] if rows else []
 return rows,info,balance
rows17=[];audit17=[];bal17=[]
for setting in SPECS17:
 for forest in SITES:
  r,a,b=match17(forest,setting);rows17.extend(r);audit17.append(a);bal17.extend(b)
PAIRS17=pd.DataFrame(rows17);AUDIT17=pd.DataFrame(audit17);BAL17=pd.DataFrame(bal17)
AUDIT17.to_csv(OUT17/'17_matching_attrition.csv',index=False)
BAL17.to_csv(OUT17/'17_covariate_balance.csv',index=False)
if len(PAIRS17):PAIRS17.to_parquet(OUT17/'17_matched_pairs.parquet',index=False)
display(AUDIT17.round(3))
if len(BAL17):display(BAL17.pivot_table(index=['site','specification'],columns='control',values='smd').round(3))


## 18 — Balance-gated NIR contrast and spatial-block uncertainty

Report the primary B8 high-minus-low reflectance contrast and 95% confidence interval only for balance-qualified forest/specification combinations. Resample 500 m spatial blocks (conditional on matching). Preserve all rejected designs in the audit, rather than silently pooling them. Spectral contrasts remain observational and cannot distinguish Musanga from other pioneer trees.


In [ ]:
def ci18(df,col,reps=2000,seed=20261120):
 groups=[g[col].to_numpy(float) for _,g in df.groupby('block500')]
 if len(groups)<MIN_BLOCKS17:return np.nan,np.nan
 sums=np.array([g.sum() for g in groups]);counts=np.array([len(g) for g in groups])
 rng=np.random.default_rng(seed);take=rng.integers(0,len(groups),(reps,len(groups)))
 dist=sums[take].sum(axis=1)/counts[take].sum(axis=1)
 return np.quantile(dist,[.025,.975])
spectral18=[]
for item in AUDIT17.itertuples():
 if not item.accepted:continue
 f=PAIRS17[(PAIRS17.site==item.site)&(PAIRS17.specification==item.specification)]
 for band in BANDS14:
  col='delta_'+band;low,high=ci18(f,col,seed=SEED17+sum(map(ord,item.site))+BANDS14.index(band))
  spectral18.append(dict(site=item.site,specification=item.specification,band=band,n_pairs=len(f),
      n_blocks=f.block500.nunique(),difference=float(f[col].mean()),ci_low=low,ci_high=high))
SPECTRAL18=pd.DataFrame(spectral18,columns=['site','specification','band','n_pairs','n_blocks','difference','ci_low','ci_high'])
SPECTRAL18.to_csv(OUT17/'18_balance_qualified_spectral.csv',index=False)
if SPECTRAL18.empty:print('NO BALANCE-QUALIFIED SPECTRAL CONTRAST. NB08 previous B8 association is not confirmed after adequate balancing.')
else:display(SPECTRAL18[SPECTRAL18.band.eq('B8')].round(5))


## 19 — Final 30 m versus 50 m strict leave-one-forest-out validation

Reuse *unchanged* cached NB08 native-scale held-out predictions, including all available test cells. Publish per-forest and forest-equal R²/MAE/RMSE, conditional cover-bin calibration, MOS high-cover bias and recall, and spatial-block bootstrap uncertainty. Scatterplot subsampling is **for display only**, balanced by forest. These are UAV-teacher validation metrics; neither independent field truth nor an independent region-wide Kisangani validation.


In [ ]:
V19=[]
for scale in [30,50]:
 for forest in SITES:
  path=SCALE_RUN/f'09_LOFO_{scale}m_{forest}.parquet'
  assert path.exists(),('Missing NB08 held-out predictions',path)
  d=pd.read_parquet(path)
  assert d.site.eq(forest).all() and d.target_fraction.between(0,1).all()
  assert d.prediction.between(0,1).all()
  d=d.copy();d['scale_m']=scale;V19.append(d)
V19=pd.concat(V19,ignore_index=True)
V19['cover_bin']=pd.cut(V19.target_fraction,[-.001,.05,.20,.40,.60,1.001],labels=['0-5%','5-20%','20-40%','40-60%','60-100%'],include_lowest=True)
metrics19=[];bins19=[];equal19=[]
for (scale,forest),g in V19.groupby(['scale_m','site']):
 y=g.target_fraction.to_numpy(float);p=g.prediction.to_numpy(float);e=p-y;high=y>=.4
 metrics19.append(dict(scale_m=scale,site=forest,n=len(g),R2=1-np.mean(e**2)/np.var(y) if np.var(y)>0 else np.nan,
  MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),bias_pp=100*np.mean(e),
  target_SD_pp=100*np.std(y),n_high=int(high.sum()),high_bias_pp=100*np.mean(e[high]) if high.any() else np.nan,
  high_recall=float(np.mean(p[high]>=.4)) if high.any() else np.nan))
 for b,z in g.groupby('cover_bin',observed=True):
  bins19.append(dict(scale_m=scale,site=forest,cover_bin=str(b),n=len(z),
    observed_pp=100*z.target_fraction.mean(),predicted_pp=100*z.prediction.mean(),
    bias_pp=100*(z.prediction-z.target_fraction).mean()))
for scale,g in V19.groupby('scale_m'):
 forests=[v for _,v in g.groupby('site')]
 mean_ref=np.mean([f.target_fraction.mean() for f in forests])
 mse=np.mean([np.mean((f.prediction-f.target_fraction)**2) for f in forests])
 variance=np.mean([np.mean((f.target_fraction-mean_ref)**2) for f in forests])
 equal19.append(dict(scale_m=scale,n_forests=len(forests),forest_equal_R2=1-mse/variance,
  forest_equal_MAE_pp=100*np.mean([np.mean(abs(f.prediction-f.target_fraction)) for f in forests]),
  forest_equal_RMSE_pp=100*np.sqrt(mse),forest_equal_target_SD_pp=100*np.sqrt(variance)))
MET19=pd.DataFrame(metrics19);BIN19=pd.DataFrame(bins19);EQ19=pd.DataFrame(equal19)
MET19.to_csv(OUT17/'19_LOFO_per_forest_30_50m.csv',index=False)
BIN19.to_csv(OUT17/'19_LOFO_cover_bins_30_50m.csv',index=False)
EQ19.to_csv(OUT17/'19_LOFO_forest_equal_30_50m.csv',index=False)
display(EQ19.round(3));display(MET19.round(3));display(BIN19.round(2))
fig,axes=plt.subplots(1,2,figsize=(13,5),sharex=True,sharey=True,layout='constrained')
for ax,scale in zip(axes,[30,50]):
 for i,forest in enumerate(SITES):
  g=V19[(V19.scale_m==scale)&(V19.site==forest)]
  sample=g.sample(n=min(len(g),2000),random_state=SEED17+scale+i)
  ax.scatter(sample.target_fraction*100,sample.prediction*100,s=6,alpha=.15,label=f'{forest} ({len(g):,})')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(xlim=(0,100),ylim=(0,100),xlabel='UAV reference cover (%)',ylabel='Satellite-predicted cover (%)',title=f'{scale} m — LOFO')
 ax.legend(fontsize=8)
fig.savefig(OUT17/'19_balanced_display_30_50m.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)
fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
for ax,forest in zip(axs.flat,SITES):
 for scale in [30,50]:
  a=BIN19[(BIN19.scale_m==scale)&(BIN19.site==forest)]
  ax.plot(a.observed_pp,a.predicted_pp,'o-',label=f'{scale} m')
 ax.plot([0,100],[0,100],'k--',lw=1)
 ax.set(title=forest,xlim=(0,100),ylim=(0,100),xlabel='Observed cover bin mean (%)',ylabel='Predicted cover bin mean (%)')
 ax.legend()
fig.savefig(OUT17/'19_cover_bin_calibration.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)


## 20 — Spatial block uncertainty and decision audit

Spatial bootstrap resamples 500 m blocks separately within each held-out forest, then forest-equalizes MAE. The 30 m and 50 m analyses have different native cell supports, so interval overlap is descriptive rather than a paired hypothesis test. This does not estimate uncertainty from UAV teacher segmentation, model-training variability or transfer to unsampled forests.


In [ ]:
rows20=[]
for scale in [30,50]:
 byforest=[];nall=0
 for forest in SITES:
  g=V19[(V19.scale_m==scale)&(V19.site==forest)].copy()
  if not {'x','y'}.issubset(g.columns):
   coords=pd.read_parquet(SCALE_RUN/f'{forest}_{scale}m.parquet',columns=['site','row','col','x','y'])
   g=g.merge(coords,on=['site','row','col'],validate='one_to_one')
  assert g[['x','y']].notna().all().all()
  g['block500']=g.x.floordiv(500).astype('int64').astype(str)+'_'+g.y.floordiv(500).astype('int64').astype(str)
  agg=g.assign(abs_error=(g.prediction-g.target_fraction).abs(),error=g.prediction-g.target_fraction).groupby('block500').agg(n=('error','size'),ae=('abs_error','sum'),err=('error','sum'))
  n=len(agg);nall+=n
  rng=np.random.default_rng(SEED17+scale+sum(map(ord,forest)))
  vals=agg[['n','ae','err']].to_numpy(float)
  draw=rng.integers(0,n,(1000,n))
  total=vals[draw].sum(axis=1)
  mae=100*total[:,1]/total[:,0];bias=100*total[:,2]/total[:,0]
  byforest.append(mae)
  rows20.append(dict(scale_m=scale,site=forest,n_blocks=n,MAE_ci_low_pp=np.quantile(mae,.025),MAE_ci_high_pp=np.quantile(mae,.975),
     bias_ci_low_pp=np.quantile(bias,.025),bias_ci_high_pp=np.quantile(bias,.975)))
 equal_dist=np.mean(byforest,axis=0)
 rows20.append(dict(scale_m=scale,site='forest_equal',n_blocks=nall,MAE_ci_low_pp=np.quantile(equal_dist,.025),
    MAE_ci_high_pp=np.quantile(equal_dist,.975),bias_ci_low_pp=np.nan,bias_ci_high_pp=np.nan))
CI20=pd.DataFrame(rows20);CI20.to_csv(OUT17/'20_spatial_bootstrap_uncertainty.csv',index=False)
display(CI20.round(3))
print('Interpretation: a significant B8 contrast requires balance-qualified matches; do not claim species-specificity.')
print('Resolution choice: consider forest-equal skill, MOS high-cover bias, target support, and spatial block intervals together.')
print('New output folder:',OUT17)


## 21 — Forest-balanced spatial-block cross-validation at 50 m (V4)

**Purpose:** quantify prediction within *represented* forests, complementing the strict leave-one-forest-out transfer experiments (Sections 01–02). Reuses the **V4 binary Musanga fractional-cover targets** in `D` and the original Sentinel-2/embedding feature families. This does not replace LOFO and does not retrain the UAV segmentation network.

- **Validation unit:** whole 500 m geographical blocks; five spatial folds, with a 100 m train–test exclusion buffer. Folds are assigned within each forest using only geographical block size and deterministic randomization, not Musanga labels.
- **Training:** maximum 1,500 cells per represented forest, stratified by cover *within training data only*. No held-out labels enter sampling or model selection.
- **Inference:** out-of-fold predictions for all eligible V4 50 m cells, compared with the **same cells** from the existing LOFO table. Equal-forest R² uses a common global mean and gives each forest equal weight (it is **not** the average of individual forest R² values).
- **Reporting:** overall and per-forest R², MAE, RMSE, bias, calibration slope/intercept, high-cover bias and recall, 500 m observed–predicted panels and bin calibration. The model is conditioned on having training observations in all four forests; its score is **not** evidence of extrapolation to an unseen forest.
- **Caveat:** 500 m blocks and the 100 m exclusion buffer are a prespecified starting point, not proof of full decorrelation. A 1 km-block sensitivity study may be needed. Reconstructed V4 inputs and model outputs must be verified in Colab before interpreting new scores.


In [ ]:
# SECTION 21 — spatially blocked 50 m CV, forest-balanced training, common-cell LOFO comparison
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression
import hashlib,json,os
assert 'D4' in globals() and 'P' in globals(), 'Run 00V4 and Sections 01–02 first'
assert D.equals(D4), 'Active target is not the Section 00V4 V4 binary-cover reference'
S21=['Baego','MOS','Yangambi','Yoko']
K21=['site','mosaic','row','col']
F21={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':list(S2)+list(EMB)}
B21=500.0
BUFFER21=100.0
FOLDS21=5
CAP21=1500
SEED21=20261123
M21=D.copy().reset_index(drop=True)
assert not M21.duplicated(K21).any()
assert M21.target_fraction.between(0,1).all()
assert set(S21)==set(M21.site.unique())
assert M21[['x','y']+list(set(sum(F21.values(),[])))].notna().all().all()
M21['block_x21']=np.floor(M21.x/B21).astype('int64')
M21['block_y21']=np.floor(M21.y/B21).astype('int64')
M21['block21']=(M21.site.astype(str)+'_'+M21.block_x21.astype(str)+'_'+M21.block_y21.astype(str))
M21['fold21']=-1
# Block-count balancing by cell numbers, without examining response or predictors.
rng21=np.random.default_rng(SEED21)
for forest in S21:
    ix=np.flatnonzero(M21.site.to_numpy()==forest)
    counts=M21.iloc[ix].groupby('block21').size()
    assert len(counts)>=FOLDS21,(forest,'Not enough spatial blocks',len(counts))
    shuffled=rng21.permutation(len(counts))
    blocks=counts.index.to_numpy()[shuffled]
    sizes=counts.to_numpy()[shuffled]
    order=np.argsort(-sizes,kind='stable')
    fold_size=np.zeros(FOLDS21,dtype=int)
    mapping={}
    for k in order:
        fold=int(np.argmin(fold_size))
        mapping[blocks[k]]=fold
        fold_size[fold]+=int(sizes[k])
    M21.loc[ix,'fold21']=M21.iloc[ix].block21.map(mapping).to_numpy(int)
assert M21.fold21.ge(0).all()
assert M21.groupby('block21').fold21.nunique().eq(1).all()
source_key21=hashlib.sha256(pd.util.hash_pandas_object(
    M21[K21+['x','y','target_fraction','fold21']],index=False).values.tobytes()).hexdigest()
CFG21=dict(version='v4_spatial_block_balanced_50m_v1',
    source_sha=SOURCE_SHA,fold_sha=source_key21,block_m=B21,buffer_m=BUFFER21,
    folds=FOLDS21,forest_cap=CAP21,seed=SEED21,models=MODEL,
    feature_sets={k:len(v) for k,v in F21.items()},
    target='NB07_V4_binary_fraction')
RUN21=V4_OUT/'25_NB08_V4_spatial_block_50m'/hashlib.sha256(
    json.dumps(CFG21,sort_keys=True).encode()).hexdigest()[:14]
RUN21.mkdir(parents=True,exist_ok=True)
cfg_file21=RUN21/'21_config.json'
if cfg_file21.exists():
    assert json.loads(cfg_file21.read_text())==CFG21,'Run configuration mismatch'
else:
    cfg_file21.write_text(json.dumps(CFG21,indent=2))
M21[K21+['x','y','block21','fold21','target_fraction']].to_parquet(
    RUN21/'21_spatial_folds.parquet',index=False)
print('Spatial blocks by forest:',M21.groupby('site').block21.nunique().to_dict())
print('Cells per fold and forest:');display(M21.groupby(['site','fold21']).size().unstack(fill_value=0))
# Each test fold has all four forests represented. Buffer removes any training
# cell closer than 100 m to a held-out cell in the SAME forest.
P21=M21[K21+['x','y','target_fraction','fold21','block21']].copy()
for name in F21:P21[name]=np.nan
AUDIT21=[]
for fold in range(FOLDS21):
    test_ix=np.flatnonzero(M21.fold21.to_numpy()==fold)
    training=[]
    audit={'fold':fold,'n_test':len(test_ix)}
    for forest in S21:
        pool=np.flatnonzero((M21.site.to_numpy()==forest)&(M21.fold21.to_numpy()!=fold))
        held=test_ix[M21.site.to_numpy()[test_ix]==forest]
        test_points=M21.iloc[held][['x','y']].to_numpy(float)
        train_points=M21.iloc[pool][['x','y']].to_numpy(float)
        if len(test_points):
            far=cKDTree(test_points).query(train_points,k=1)[0]>=BUFFER21
            pool=pool[far]
        assert len(pool)>0,(fold,forest,'Buffer leaves no training cells')
        rng=np.random.default_rng(SEED21+fold*101+sum(map(ord,forest)))
        if len(pool)>CAP21:
            ytrain=M21.iloc[pool].target_fraction.to_numpy(float)
            strata=np.digitize(ytrain,[.10,.20,.40])
            take=[]
            for k in range(4):
                group=pool[strata==k]
                if len(group):
                    n=min(len(group),CAP21//4)
                    take.extend(rng.choice(group,n,replace=False).tolist())
            if len(take)<CAP21:
                leftover=np.setdiff1d(pool,np.array(take,dtype=int),assume_unique=False)
                take.extend(rng.choice(leftover,min(CAP21-len(take),len(leftover)),replace=False).tolist())
            pool=np.array(sorted(take),dtype=int)
        training.extend(pool.tolist())
        audit[forest+'_train']=len(pool)
        audit[forest+'_test']=len(held)
    train_ix=np.asarray(training,dtype=int)
    assert not np.intersect1d(train_ix,test_ix).size
    assert set(M21.iloc[train_ix].block21).isdisjoint(set(M21.iloc[test_ix].block21))
    for name,features in F21.items():
        cache21=RUN21/f'fold{fold}_{name}.npz'
        if cache21.is_file():
            with np.load(cache21,allow_pickle=False) as z:
                assert z['source_sha'].item()==source_key21
                assert np.array_equal(z['test_ix'],test_ix)
                pred=z['pred']
            print('CACHE 21',fold,name,flush=True)
        else:
            model=HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
            model.fit(M21.iloc[train_ix][features].to_numpy(np.float32),
                      M21.iloc[train_ix].target_fraction.to_numpy(np.float32))
            pred=np.clip(model.predict(M21.iloc[test_ix][features].to_numpy(np.float32)),0,1)
            temp=cache21.with_suffix('.partial')
            with open(temp,'wb') as fd:
                np.savez_compressed(fd,source_sha=source_key21,test_ix=test_ix,pred=pred)
            os.replace(temp,cache21)
            print('FIT 21',fold,name,'test',len(test_ix),flush=True)
        P21.loc[test_ix,name]=pred
    AUDIT21.append(audit)
assert P21[list(F21)].notna().all().all()
A21=pd.DataFrame(AUDIT21)
A21.to_csv(RUN21/'21_fold_training_audit.csv',index=False)
P21.to_parquet(RUN21/'21_spatial_block_OOF_predictions.parquet',index=False)
display(A21)

def summaries21(frame,pred_names,validation):
    rows=[]
    for name in pred_names:
        groups=list(frame.groupby('site'))
        overall=float(np.mean([g.target_fraction.mean() for _,g in groups]))
        pooled_mse=np.mean([np.mean((g[name]-g.target_fraction)**2) for _,g in groups])
        pooled_var=np.mean([np.mean((g.target_fraction-overall)**2) for _,g in groups])
        site_rows=[]
        for forest,g in groups:
            y=g.target_fraction.to_numpy(float);p=g[name].to_numpy(float)
            e=p-y;hi=y>=.4
            fit=LinearRegression().fit(y.reshape(-1,1),p) if np.var(y)>1e-10 else None
            site_rows.append(dict(validation=validation,model=name,site=forest,n=len(g),
                R2=1-np.mean(e**2)/np.var(y) if np.var(y)>1e-10 else np.nan,
                MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),
                bias_pp=100*np.mean(e),calibration_slope=fit.coef_[0] if fit else np.nan,
                calibration_intercept_pp=100*fit.intercept_ if fit else np.nan,
                high_n=int(hi.sum()),high_bias_pp=100*np.mean(e[hi]) if hi.any() else np.nan,
                high_recall=np.mean(p[hi]>=.4) if hi.any() else np.nan))
        rows.extend(site_rows)
        rows.append(dict(validation=validation,model=name,site='forest_equal',n=len(frame),
            R2=1-pooled_mse/pooled_var if pooled_var>0 else np.nan,
            MAE_pp=np.mean([r['MAE_pp'] for r in site_rows]),
            RMSE_pp=np.sqrt(np.mean([r['RMSE_pp']**2 for r in site_rows])),
            bias_pp=np.mean([r['bias_pp'] for r in site_rows]),
            calibration_slope=np.nan,calibration_intercept_pp=np.nan,
            high_n=sum(r['high_n'] for r in site_rows),
            high_bias_pp=np.nan,high_recall=np.nan))
    return pd.DataFrame(rows)

# Match LOFO to precisely the same V4 50m geographical keys.
LOFO21=P[K21+list(F21)].copy()
PAIR21=P21.merge(LOFO21,on=K21,validate='one_to_one',suffixes=('_block','_lofo'))
assert len(PAIR21)==len(P21),(len(PAIR21),len(P21))
B21P=PAIR21[K21+['target_fraction']+[name+'_block' for name in F21]].copy()
L21P=PAIR21[K21+['target_fraction']+[name+'_lofo' for name in F21]].copy()
B21P=B21P.rename(columns={name+'_block':name for name in F21})
L21P=L21P.rename(columns={name+'_lofo':name for name in F21})
MET21=pd.concat([summaries21(B21P,list(F21),'spatial_block'),
                 summaries21(L21P,list(F21),'LOFO')],ignore_index=True)
MET21.to_csv(RUN21/'21_matched_LOFO_vs_spatial_block_metrics.csv',index=False)
PAIR21.to_parquet(RUN21/'21_matched_LOFO_vs_spatial_block_predictions.parquet',index=False)
display(MET21[MET21.site.eq('forest_equal')][
    ['validation','model','n','R2','MAE_pp','RMSE_pp','bias_pp']].round(3))
display(MET21[MET21.site.ne('forest_equal')][
    ['validation','model','site','R2','MAE_pp','bias_pp','high_bias_pp','high_recall']].round(3))

# Balanced display is NOT balanced performance estimation. All held-out cells
# contribute to the per-forest scores and each forest has equal weight.
rng=np.random.default_rng(SEED21+77)
draws=[]
for forest in S21:
    sub=PAIR21[PAIR21.site.eq(forest)]
    draws.append(sub.sample(n=min(175,len(sub)),random_state=SEED21+sum(map(ord,forest))))
SHOW21=pd.concat(draws,ignore_index=True)
colors21=dict(zip(S21,['#236e85','#c17930','#619e65','#9461a5']))
fig,axs=plt.subplots(2,3,figsize=(15,9),sharex=True,sharey=True,layout='constrained')
for row,method in enumerate(['block','lofo']):
    for col,model_name in enumerate(F21):
        ax=axs[row,col]
        for forest in S21:
            sub=SHOW21[SHOW21.site.eq(forest)]
            ax.scatter(sub.target_fraction*100,sub[model_name+'_'+method]*100,
                       s=13,alpha=.48,c=colors21[forest],label=forest)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),
               title=('Spatial-block CV' if row==0 else 'LOFO')+' — '+model_name,
               xlabel='UAV Musanga cover (%)',ylabel='Predicted cover (%)')
        ax.grid(alpha=.14)
h,l=axs[0,0].get_legend_handles_labels()
fig.legend(h,l,loc='upper center',ncol=4)
fig.savefig(RUN21/'21_balanced_scatter_spatial_block_vs_LOFO.png',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)

bins21=[-.001,.05,.20,.40,.60,1.001]
binlabels21=['0–5','5–20','20–40','40–60','60–100']
cal21=[]
for forest,group in PAIR21.groupby('site'):
    for method in ('block','lofo'):
        for model in F21:
            q=group.copy()
            q['bin']=pd.cut(q.target_fraction,bins21,labels=binlabels21,include_lowest=True)
            for band,g in q.groupby('bin',observed=True):
                cal21.append(dict(site=forest,validation=method,model=model,cover_bin=str(band),
                    n=len(g),observed_pp=100*g.target_fraction.mean(),
                    predicted_pp=100*g[model+'_'+method].mean(),
                    bias_pp=100*(g[model+'_'+method]-g.target_fraction).mean()))
CAL21=pd.DataFrame(cal21)
CAL21.to_csv(RUN21/'21_cover_bin_calibration.csv',index=False)
fig,ax=plt.subplots(figsize=(8,6))
for forest in S21:
    for method,style in [('block','-'),('lofo','--')]:
        part=CAL21[(CAL21.site==forest)&(CAL21.validation==method)&(CAL21.model=='M1_S2')]
        ax.plot(part.observed_pp,part.predicted_pp,marker='o',linestyle=style,
                color=colors21[forest],label=forest+' '+method)
ax.plot([0,100],[0,100],'k:',label='1:1')
ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed Musanga cover (%)',
       ylabel='Mean predicted cover (%)',title='50 m S2: forest-wise cover-bin calibration')
ax.legend(fontsize=8,ncol=2);ax.grid(alpha=.2)
fig.savefig(RUN21/'21_50m_S2_cover_bin_calibration.png',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)
print('Section 21 complete. Outputs:',RUN21)


## 22 — MOS continuous, structure-controlled spectral response

**Purpose:** test whether Sentinel-2 B8 reflectance rises continuously with V4 Musanga cover or flattens at high cover, while adjusting for the **same independent structure and disturbance covariates** used in Sections 14–18. Uses all MOS cells with valid V4 cover, spectral bands and structure controls (not only extreme high–low matched pairs).

A prespecified quadratic spline of Musanga fractional cover and additive standardized structure controls is fitted to B8, selected red-edge and SWIR bands with ridge regularization. Conditional response curves are predicted at the observed average control values; spatial uncertainty is estimated by resampling whole 500 m blocks. Both unadjusted abundance-bin means and adjusted continuous curves are shown. The 5→40% and 40→65% contrasts assess whether the spectral response weakens across abundance.

**Important:** observational conditional association, not causation or species-specific spectral identification. Forest structure may be partly downstream of Musanga abundance. Additive adjustment cannot remove unmeasured confounding or account for UAV reference uncertainty. Response curves are restricted to observed cover support; bootstrapping represents spatial clustering only.


In [ ]:
# SECTION 22 — Continuous structure-controlled spectral response at MOS
# Adjusted NIR/red-edge/SWIR curves; spatial-block bootstrap for conditional association.
from sklearn.preprocessing import SplineTransformer,StandardScaler
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr
assert 'D14' in globals() and 'CONTROLS14' in globals() and 'spec14' in globals(), 'Run Sections 14 and 17 first'
assert 'D4' in globals() and D.equals(D4), 'This must use the V4 NB07 binary-cover reference'
SITE22='MOS'
SEED22=20261123
BOOT22=300
BLOCK22=500.
ALPHA22=10.
KNOTS22=np.array([0.,.10,.20,.40,.60,.80,1.],dtype=float).reshape(-1,1)
COL22=['B8','B5','B6','B7','B8A','B11','B12']
COV22=list(CONTROLS14)
M22=D14[D14.site.eq(SITE22)].copy().replace([np.inf,-np.inf],np.nan)
M22=M22.dropna(subset=['target_fraction','x','y']+COV22+[spec14[b] for b in COL22]).reset_index(drop=True)
assert len(M22)>=100 and M22.target_fraction.between(0,1).all()
assert (M22.target_fraction>=.4).sum()>=30, 'Not enough high-cover MOS cells for continuous response'
M22['block22']=np.floor(M22.x/BLOCK22).astype('int64').astype(str)+'_'+np.floor(M22.y/BLOCK22).astype('int64').astype(str)
BLOCKS22=list(M22.groupby('block22').indices.values())
assert len(BLOCKS22)>=15,('Too few independent spatial blocks',len(BLOCKS22))
sha22=hashlib.sha256(pd.util.hash_pandas_object(
    M22[['site','mosaic','row','col','target_fraction']+COV22+[spec14[b] for b in COL22]],
    index=False).values.tobytes()).hexdigest()
CFG22=dict(version='V4_MOS_continuous_conditional_spectral_v1',
    source_sha=SOURCE_SHA,matched_sample_sha=sha22,
    block_m=BLOCK22,bootstraps=BOOT22,ridge_alpha=ALPHA22,
    spline_knots=KNOTS22.ravel().tolist(),covariates=COV22,bands=COL22,
    inferential_scope='conditional association; not species-specific causal attribution')
RUN22=V4_OUT/'26_NB08_MOS_continuous_spectral_response'/hashlib.sha256(
    json.dumps(CFG22,sort_keys=True).encode()).hexdigest()[:14]
RUN22.mkdir(parents=True,exist_ok=True)
config22=RUN22/'22_config.json'
if config22.exists():assert json.loads(config22.read_text())==CFG22
else:config22.write_text(json.dumps(CFG22,indent=2))
# Fixed spline basis and standardization across block-bootstrap replicates.
# Predict at the same measured covariate mean. This additive model does not
# impose a monotonic response; flattening must be supported by intervals.
spline22=SplineTransformer(knots=KNOTS22,degree=2,include_bias=False,
                           extrapolation='constant')
ycover22=M22.target_fraction.to_numpy(float)
sp22=spline22.fit_transform(ycover22.reshape(-1,1))
standard22=StandardScaler()
xcov22=standard22.fit_transform(M22[COV22].to_numpy(float))
X22=np.column_stack([sp22,xcov22])
Y22=M22[[spec14[b] for b in COL22]].to_numpy(float)
assert np.isfinite(X22).all() and np.isfinite(Y22).all()
low22=max(0.,float(np.quantile(ycover22,.01)))
high22=min(1.,float(np.quantile(ycover22,.99)))
assert high22>.65,(low22,high22,'Insufficient high-cover support')
grid22=np.linspace(low22,high22,81)
Xgrid22=np.column_stack([
    spline22.transform(grid22.reshape(-1,1)),
    np.zeros((len(grid22),len(COV22)),dtype=float)])
def fit22(indices):
    reg=Ridge(alpha=ALPHA22,fit_intercept=True)
    reg.fit(X22[indices],Y22[indices])
    return reg.predict(Xgrid22)
point22=fit22(np.arange(len(M22)))
rng22=np.random.default_rng(SEED22)
boot22=np.empty((BOOT22,len(grid22),len(COL22)),dtype=np.float32)
for rep in range(BOOT22):
    draw=rng22.integers(0,len(BLOCKS22),size=len(BLOCKS22))
    ids=np.concatenate([BLOCKS22[j] for j in draw])
    boot22[rep]=fit22(ids)
    if (rep+1)%100==0:print('MOS spectral block bootstrap',rep+1,'/',BOOT22,flush=True)
l22=np.quantile(boot22,.025,axis=0)
u22=np.quantile(boot22,.975,axis=0)
rows22=[]
for k,band in enumerate(COL22):
    for t,value in enumerate(grid22):
        rows22.append(dict(band=band,cover_fraction=value,cover_pct=100*value,
            adjusted_reflectance=point22[t,k],ci_low=l22[t,k],ci_high=u22[t,k]))
CURVE22=pd.DataFrame(rows22)
CURVE22.to_csv(RUN22/'22_MOS_adjusted_spectral_response.csv',index=False)
# Observed-bin diagnostics, not treated as fitted adjusted effects.
bins22=[-.001,.05,.20,.40,.60,1.001]
labels22=['0-5','5-20','20-40','40-60','60-100']
M22['abundance_bin22']=pd.cut(ycover22,bins22,labels=labels22,include_lowest=True)
BIN22=M22.groupby('abundance_bin22',observed=True).agg(
    n=('target_fraction','size'),mean_cover=('target_fraction','mean'),
    mean_B8=(spec14['B8'],'mean'),sd_B8=(spec14['B8'],'std')).reset_index()
BIN22.to_csv(RUN22/'22_MOS_observed_abundance_bins.csv',index=False)
display(BIN22)
# Contrasts from the same regression curve; uncertainty uses block bootstrap.
def contrast22(a,b,band='B8'):
    assert low22<=a<=high22 and low22<=b<=high22,(a,b,low22,high22)
    j=COL22.index(band)
    # Interpolate every bootstrap curve on identical cover grid.
    delta=float(np.interp(b,grid22,point22[:,j])-np.interp(a,grid22,point22[:,j]))
    empirical=np.array([np.interp(b,grid22,v[:,j])-np.interp(a,grid22,v[:,j])
                        for v in boot22],dtype=float)
    lo,hi=np.quantile(empirical,[.025,.975])
    return dict(band=band,from_pct=a*100,to_pct=b*100,
                adjusted_difference=delta,ci_low=lo,ci_high=hi,
                n=len(M22),n_blocks=len(BLOCKS22))
CONTRAST22=pd.DataFrame([contrast22(a,b,band)
    for band in COL22 for a,b in [(.05,.40),(.40,.65)]])
CONTRAST22.to_csv(RUN22/'22_MOS_continuous_response_contrasts.csv',index=False)
display(CONTRAST22[CONTRAST22.band.eq('B8')].round(5))
fig,ax=plt.subplots(figsize=(9,6))
ix=COL22.index('B8')
ax.plot(grid22*100,point22[:,ix],color='#185e78',label='Structure-adjusted B8')
ax.fill_between(grid22*100,l22[:,ix],u22[:,ix],alpha=.22,color='#185e78',
                label='500 m block bootstrap 95% CI')
ax.scatter(BIN22.mean_cover*100,BIN22.mean_B8,s=BIN22.n.clip(5,100),
           color='#bc762b',edgecolor='black',zorder=4,label='Observed cover-bin means')
ax.set(xlabel='UAV V4 Musanga cover (%)',ylabel='Sentinel-2 B8 reflectance',
       title='MOS — conditional continuous B8 spectral response')
ax.grid(alpha=.2);ax.legend()
fig.savefig(RUN22/'22_MOS_B8_continuous_adjusted_response.png',dpi=240,bbox_inches='tight')
plt.show();plt.close(fig)
fig,axs=plt.subplots(2,3,figsize=(14,8),layout='constrained')
for ax,band in zip(axs.flat,[b for b in COL22 if b!='B8']):
    k=COL22.index(band)
    ax.plot(grid22*100,point22[:,k],color='#185e78')
    ax.fill_between(grid22*100,l22[:,k],u22[:,k],alpha=.2,color='#185e78')
    ax.set(xlabel='Musanga cover (%)',ylabel='Reflectance',title=band)
    ax.grid(alpha=.15)
fig.savefig(RUN22/'22_MOS_rededge_SWIR_conditional_response.png',
            dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)
QA22=dict(n_cells=int(len(M22)),n_blocks=len(BLOCKS22),
          high_cover_cells=int((ycover22>=.4).sum()),
          cover_min=float(ycover22.min()),cover_max=float(ycover22.max()),
          range_plotted=[float(low22),float(high22)],forest=SITE22,
          target='NB07 V4 band 2 fraction',
          caveat='Conditional additive association; controls may be downstream of Musanga; uncertainty does not include UAV segmentation/reference error.')
(RUN22/'22_interpretation_and_QA.json').write_text(json.dumps(QA22,indent=2))
print('Section 22 complete:',RUN22)
print('Interpretation: association conditional on measured structure and disturbance; not causal attribution.')


## 23 — Does abundance-aware modelling improve 50 m fractional cover?

**Purpose:** exploratory comparison on the *exact fixed spatial folds and V4 targets* from Section 21, without changing the original LOFO or block-CV results.

**A. Abundance-weighted regression:** moderate and strong weights for abundant canopy cover, normalized so each training forest contributes equal total sample weight; retains the existing Section 21 forest cap. **B. Probabilistic two-stage:** four cover classes (0–5%, 5–20%, 20–40%, ≥40%); soft class probabilities multiplied by class-conditional cover regressions, without a hard class boundary. **C. Leakage-controlled nonlinear calibration:** each *outer* 500 m spatial fold is calibrated by isotonic regression using only four *inner*, spatially held-out folds from its training forests. Neither calibration fit nor inner base models use outer test responses.

Models: S2 annual/quarterly and S2+embeddings. Metrics use all outer held-out cells and equal forest aggregation. Outputs include cover-bin calibration (especially MOS above 60%), low-cover error, paired block-bootstrap MAE differences and scatters. The baseline Section 21 out-of-fold predictions are reused exactly. No extra tuning on outer held-out labels.

**Cautions:** exploratory model comparison after inspecting earlier folds; a selected winner needs independent outer validation. Upweighting may worsen false positives. Two-stage modelling cannot create absent spectral information. Isotonic calibration cannot restore missing spatial discrimination. The underlying folds have uneven forest training counts despite equal total sample weights.

In [ ]:
# SECTION 23 — abundance-weighting, probabilistic two-stage, nested spatial calibration
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from scipy.spatial import cKDTree
assert 'M21' in globals() and 'P21' in globals() and 'RUN21' in globals(), 'Run Section 21 first'
assert 'summaries21' in globals()
S23=['Baego','MOS','Yangambi','Yoko']
SETS23={'S2':list(S2),'S2_EMB':list(S2)+list(EMB)}
SEED23=20261124
CAP23=1500
WEIGHTS23={'moderate':[1.,1.,1.5,2.,3.],
           'strong':[1.,1.,2.,3.,5.]}
STRATEGIES23=['baseline','moderate','strong','two_stage','nested_isotonic']
K23=['site','mosaic','row','col']
assert M21[K23].equals(P21[K23])
assert len(M21)==10639 and set(M21.site)==set(S23)
assert M21.fold21.nunique()==5
CFG23=dict(version='v4_abundance_spatial_sensitivity_v2',source_sha=SOURCE_SHA,
           folds_hash=source_key21,seed=SEED23,cap=CAP23,
           weights=WEIGHTS23,features={k:len(v) for k,v in SETS23.items()},
           calibration='four inner block folds nested in each outer fold')
RUN23=V4_OUT/'27_NB08_V4_abundance_sensitivity_50m'/hashlib.sha256(
    json.dumps(CFG23,sort_keys=True).encode()).hexdigest()[:14]
RUN23.mkdir(parents=True,exist_ok=True)
p23=RUN23/'23_config.json'
if p23.exists():assert json.loads(p23.read_text())==CFG23
else:p23.write_text(json.dumps(CFG23,indent=2))
Y23=M21.target_fraction.to_numpy(float)
SITE23=M21.site.to_numpy()
FOLD23=M21.fold21.to_numpy()
XY23=M21[['x','y']].to_numpy(float)
def pick23(pool,forest,fold):
    if len(pool)<=CAP23:return pool
    # Exactly reproduce Section 21's training cap sampling.
    rng=np.random.default_rng(SEED21+fold*101+sum(map(ord,forest)))
    strata=np.digitize(Y23[pool],[.10,.20,.40])
    chosen=[]
    for k in range(4):
        ids=pool[strata==k]
        if len(ids):chosen.extend(rng.choice(ids,min(len(ids),CAP23//4),replace=False).tolist())
    if len(chosen)<CAP23:
        rest=np.setdiff1d(pool,np.array(chosen,dtype=int),assume_unique=False)
        chosen.extend(rng.choice(rest,CAP23-len(chosen),replace=False).tolist())
    return np.array(sorted(chosen),dtype=int)
def training23(excluded_folds,validation_idx,fold):
    train=[]
    counts={}
    for forest in S23:
        pool=np.flatnonzero((SITE23==forest)&(~np.isin(FOLD23,excluded_folds)))
        val=validation_idx[SITE23[validation_idx]==forest]
        if len(val):
            distance=cKDTree(XY23[val]).query(XY23[pool],k=1)[0]
            pool=pool[distance>=100.]
        assert len(pool)>0,(forest,fold,'No spatially buffered training cells')
        pool=pick23(pool,forest,fold)
        train.extend(pool.tolist())
        counts[forest]=len(pool)
    train=np.asarray(train,dtype=int)
    assert not np.intersect1d(train,validation_idx).size
    assert set(M21.iloc[train].block21).isdisjoint(set(M21.iloc[validation_idx].block21))
    return train,counts
def balanced_weights23(labels,forests,profile=None):
    weights=np.ones(len(labels),dtype=float)
    if profile is not None:
        bins=np.digitize(labels,[.05,.20,.40,.60])
        weights=np.asarray(WEIGHTS23[profile],dtype=float)[bins]
    for forest in S23:
        mask=forests==forest
        if mask.any():weights[mask]/=weights[mask].sum()
    return weights*len(weights)/len(S23)
def cached23(path,test_idx,fit_function):
    if path.is_file():
        with np.load(path,allow_pickle=False) as z:
            assert np.array_equal(z['test_idx'],test_idx)
            assert z['source_sha'].item()==source_key21
            return z['prediction']
    prediction=np.asarray(fit_function(),dtype=float)
    assert len(prediction)==len(test_idx) and np.isfinite(prediction).all()
    partial=path.with_suffix('.partial')
    with open(partial,'wb') as fh:
        np.savez_compressed(fh,test_idx=test_idx,source_sha=source_key21,
                            prediction=prediction)
    os.replace(partial,path)
    return prediction
P23=M21[K23+['x','y','target_fraction','fold21','block21']].copy()
for fs in SETS23:
    for strategy in STRATEGIES23:P23[fs+'__'+strategy]=np.nan
audit23=[]
for outer in range(5):
    test=np.flatnonzero(FOLD23==outer)
    train,counts=training23([outer],test,outer)
    audit23.append(dict(fold=outer,n_test=len(test),**counts))
    for fs,features in SETS23.items():
        baseline_col={'S2':'M1_S2','S2_EMB':'M3_S2_EMB'}[fs]
        base=P21.iloc[test][baseline_col].to_numpy(float)
        P23.loc[test,fs+'__baseline']=base
        Xtr=M21.iloc[train][features].to_numpy(np.float32)
        Xte=M21.iloc[test][features].to_numpy(np.float32)
        yt=Y23[train];st=SITE23[train]
        for scenario in WEIGHTS23:
            def train_weighted23():
                model=HistGradientBoostingRegressor(random_state=SEED23,**MODEL)
                model.fit(Xtr,yt,sample_weight=balanced_weights23(yt,st,scenario))
                return np.clip(model.predict(Xte),0,1)
            f=RUN23/f'{fs}_{scenario}_outer{outer}.npz'
            pred=cached23(f,test,train_weighted23)
            P23.loc[test,fs+'__'+scenario]=pred
            print('DONE',outer,fs,scenario,flush=True)
        def train_two_stage23():
            classes=np.digitize(yt,[.05,.20,.40])
            classifier=HistGradientBoostingClassifier(random_state=SEED23,**MODEL)
            classifier.fit(Xtr,classes,sample_weight=balanced_weights23(yt,st,'moderate'))
            assert np.array_equal(classifier.classes_,np.arange(4))
            probs=classifier.predict_proba(Xte)
            conditional=np.empty_like(probs)
            bounds=[(0,.05),(.05,.20),(.20,.40),(.40,1.)]
            for c,(low,high) in enumerate(bounds):
                ix=classes==c
                if ix.sum()>=35 and np.unique(yt[ix]).size>=10:
                    r=HistGradientBoostingRegressor(random_state=SEED23+c,**MODEL)
                    r.fit(Xtr[ix],yt[ix],sample_weight=balanced_weights23(
                        yt[ix],st[ix],'moderate'))
                    conditional[:,c]=np.clip(r.predict(Xte),low,high)
                else:
                    conditional[:,c]=float(yt[ix].mean()) if ix.any() else (low+high)/2
            return np.clip(np.sum(probs*conditional,axis=1),0,1)
        P23.loc[test,fs+'__two_stage']=cached23(
            RUN23/f'{fs}_two_stage_outer{outer}.npz',test,train_two_stage23)
        print('DONE',outer,fs,'two_stage',flush=True)
        def train_nested_cal23():
            # Outer test observations NEVER participate in inner training or calibration.
            inner_predictions=[];inner_truth=[];inner_site=[]
            for inner in range(5):
                if inner==outer:continue
                val=np.flatnonzero(FOLD23==inner)
                itr,_=training23([outer,inner],val,inner)
                h=HistGradientBoostingRegressor(random_state=SEED21,**MODEL)
                h.fit(M21.iloc[itr][features].to_numpy(np.float32),Y23[itr])
                inner_predictions.append(np.clip(h.predict(
                    M21.iloc[val][features].to_numpy(np.float32)),0,1))
                inner_truth.append(Y23[val]);inner_site.append(SITE23[val])
            px=np.concatenate(inner_predictions);true=np.concatenate(inner_truth)
            sites=np.concatenate(inner_site)
            cal=IsotonicRegression(y_min=0.,y_max=1.,out_of_bounds='clip')
            cal.fit(px,true,sample_weight=balanced_weights23(true,sites))
            return np.clip(cal.predict(base),0,1)
        P23.loc[test,fs+'__nested_isotonic']=cached23(
            RUN23/f'{fs}_nested_isotonic_outer{outer}.npz',test,train_nested_cal23)
        print('DONE',outer,fs,'nested_isotonic',flush=True)
pd.DataFrame(audit23).to_csv(RUN23/'23_training_audit.csv',index=False)
names23=[fs+'__'+method for fs in SETS23 for method in STRATEGIES23]
assert P23[names23].notna().all().all()
P23.to_parquet(RUN23/'23_outer_fold_predictions.parquet',index=False)
MET23=summaries21(P23,names23,'spatial_block_50m')
MET23['feature_set']=MET23.model.str.split('__').str[0]
MET23['strategy']=MET23.model.str.split('__').str[1]
MET23.to_csv(RUN23/'23_scores_by_forest.csv',index=False)
print('Forest-equal full-heldout performance:')
display(MET23[MET23.site.eq('forest_equal')][
    ['feature_set','strategy','R2','MAE_pp','RMSE_pp','bias_pp']].round(3))
print('MOS high-cover diagnostics:')
display(MET23[MET23.site.eq('MOS')][
    ['feature_set','strategy','R2','MAE_pp','high_bias_pp','high_recall']].round(3))
# Paired block bootstrap for change in MAE. Do not treat post hoc best
# method as independently validated; these are exploratory comparisons.
delta_rows23=[]
for fs in SETS23:
    base=np.abs(P23[fs+'__baseline'].to_numpy(float)-Y23)*100
    for strategy in STRATEGIES23[1:]:
        delta=100*np.abs(P23[fs+'__'+strategy].to_numpy(float)-Y23)-base
        point=np.mean([delta[SITE23==s].mean() for s in S23])
        rng=np.random.default_rng(SEED23+sum(map(ord,fs+strategy)))
        pools={}
        for forest in S23:
            idx=np.flatnonzero(SITE23==forest)
            pools[forest]=[idx[v] for v in P23.iloc[idx].groupby('block21').indices.values()]
        bs=[]
        for _ in range(300):
            m=[]
            for forest in S23:
                groups=pools[forest]
                draws=rng.integers(0,len(groups),size=len(groups))
                ids=np.concatenate([groups[j] for j in draws])
                m.append(delta[ids].mean())
            bs.append(np.mean(m))
        low,high=np.quantile(bs,[.025,.975])
        delta_rows23.append(dict(features=fs,strategy=strategy,
                                 delta_MAE_pp=point,ci_low=low,ci_high=high))
DIFF23=pd.DataFrame(delta_rows23)
DIFF23.to_csv(RUN23/'23_paired_block_MAE_differences.csv',index=False)
display(DIFF23.round(3))
CAL23=[]
for fs in SETS23:
    for method in STRATEGIES23:
        g=P23[P23.site.eq('MOS')].copy()
        g['cover_class']=pd.cut(g.target_fraction,[-.001,.05,.20,.40,.60,1.001],
                    labels=['0-5','5-20','20-40','40-60','60-100'],include_lowest=True)
        for label,sub in g.groupby('cover_class',observed=True):
            CAL23.append(dict(features=fs,strategy=method,cover_class=str(label),
                              n=len(sub),observed_pp=100*sub.target_fraction.mean(),
                              predicted_pp=100*sub[fs+'__'+method].mean(),
                              bias_pp=100*(sub[fs+'__'+method]-sub.target_fraction).mean()))
CAL23=pd.DataFrame(CAL23)
CAL23.to_csv(RUN23/'23_MOS_cover_bin_calibration.csv',index=False)
for fs in SETS23:
    fig,ax=plt.subplots(figsize=(8.5,6))
    for method in STRATEGIES23:
        g=CAL23[(CAL23.features==fs)&(CAL23.strategy==method)]
        ax.plot(g.observed_pp,g.predicted_pp,'o-',label=method)
    ax.plot([0,100],[0,100],'k:',label='1:1')
    ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed MOS V4 cover (%)',
           ylabel='Predicted cover (%)',title='MOS — '+fs+' spatial-block calibration')
    ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.savefig(RUN23/('23_MOS_calibration_'+fs+'.png'),dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
    fig,ax=plt.subplots(figsize=(7,6))
    for forest in S23:
        g=P23[P23.site.eq(forest)].sample(n=min(175,(SITE23==forest).sum()),
                         random_state=SEED23+sum(map(ord,forest)))
        ax.scatter(100*g.target_fraction,100*g[fs+'__moderate'],
                   s=13,alpha=.5,label=forest)
    ax.plot([0,100],[0,100],'k--',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed V4 cover (%)',
           ylabel='Predicted cover (%)',title=fs+' — moderate weighting')
    ax.legend();ax.grid(alpha=.2)
    fig.savefig(RUN23/('23_balanced_scatter_'+fs+'.png'),dpi=220,bbox_inches='tight')
    plt.show();plt.close(fig)
print('Section 23 outputs:',RUN23)
print('Exploratory comparison: model selection needs new independent spatial validation.')


## 24 — Does the MOS spectral response actually flatten?

**D. Formal sensitivity diagnostic, complementing Section 22.** Reuses the same 300 spatial-block bootstrap replicates from the structure-controlled MOS spline analysis. Compares the **per-cover-unit** B8, red-edge and SWIR response slopes over 5–40% and 40–65% Musanga cover and evaluates a paired bootstrap interval for their difference. A negative difference indicates a weaker adjusted response at high cover; a confidence interval including zero does not establish flattening. Reflectance-response flattening and regression compression are related but distinct hypotheses.

Run Section 22 before Section 24. Spatial uncertainty excludes V4 UAV-reference and spectral measurement uncertainty; controls may lie on causal pathways. Results are observational associations, not species-specific causal effects.

In [ ]:
# SECTION 24 — explicit slope-change test from Section 22 spatial-block replicates
assert 'boot22' in globals() and 'point22' in globals() and 'grid22' in globals(), 'Run Section 22 first'
assert 'COL22' in globals() and 'RUN22' in globals() and 'RUN23' in globals()
SLOPE24=[]
for band in COL22:
    k=COL22.index(band)
    def slopes(curve):
        a=np.interp(.05,grid22,curve[:,k])
        b=np.interp(.40,grid22,curve[:,k])
        c=np.interp(.65,grid22,curve[:,k])
        lower=(b-a)/.35
        upper=(c-b)/.25
        return lower,upper,upper-lower
    vals=np.array([slopes(curve) for curve in boot22])
    estimate=slopes(point22)
    ci=np.quantile(vals[:,2],[.025,.975])
    SLOPE24.append(dict(band=band,n_blocks=len(BLOCKS22),
        low_slope_per_fraction=estimate[0],
        high_slope_per_fraction=estimate[1],
        high_minus_low_slope=estimate[2],
        ci_low=ci[0],ci_high=ci[1],
        bootstrap_proportion_negative=float(np.mean(vals[:,2]<0))))
SLOPE24=pd.DataFrame(SLOPE24)
SLOPE24.to_csv(RUN23/'24_MOS_spatial_bootstrap_slope_change.csv',index=False)
display(SLOPE24.round(5))
fig,ax=plt.subplots(figsize=(8,4.7))
rows=SLOPE24.set_index('band').loc[COL22]
y=np.arange(len(rows))
ax.errorbar(rows.high_minus_low_slope,y,
    xerr=np.vstack([rows.high_minus_low_slope-rows.ci_low,
                    rows.ci_high-rows.high_minus_low_slope]),
    fmt='o',capsize=3)
ax.axvline(0,color='black',linestyle=':')
ax.set(yticks=y,yticklabels=rows.index,
       xlabel='Adjusted slope difference: high (40–65%) minus low (5–40%)',
       title='MOS — block bootstrap test of spectral response flattening')
ax.grid(axis='x',alpha=.2)
fig.savefig(RUN23/'24_MOS_spectral_slope_change.png',dpi=220,bbox_inches='tight')
plt.show();plt.close(fig)
print('Section 24 complete: changes relative to Section 22 estimates, not causal effects.')
